# Proof of Concept – Instant RAG (ADD/MUL als Proxy-Tools)

Bachelorarbeit Roland Kummer (HS Offenburg × t2b AG). Dieses Notebook dokumentiert alle PoC-Läufe gegen den
unveränderten Referenz-Harness `src/instantrag_qwen_demo.py`. Plan, Entscheidungen und Status stehen in
`doc/poc-sprint-plan.md`.

**Regeln für dieses Notebook**
- Der Harness wird nur importiert und von außen beobachtet, nie verändert.
- Rohdaten sind append-only (`results/runs.jsonl`); der Offline-Selbsttest schreibt nach `results/selftest.jsonl`.
- `model_cfg` wird nie ausgegeben (enthält den API-Key), Request-Header werden nie geloggt.
- Zellen, die Requests an das LLM auslösen, stehen standardmäßig auf `RUN = False`.

## 0. Zweck, Hypothesen, fixierte Schwellen

*Gliederungspunkt – wird vor E1 ausgearbeitet.* Hier stehen Zweck des PoC, die Hypothesen H1–H9
(Plan §6) und die am 15.11.2026 entschiedenen Erfolgsschwellen für H1 (Plan §2, O6) mit Datum.

## 1. Einrichtung

Pfade, Import des Harness, Konfiguration (ohne Key), Metadaten, JSONL-Writer, Rekorder, Darstellung und
Offline-Selbsttest, Prompt-Versionen S0–S2 (1.4, Register `doc/prompt-versions.md`) und die eingefrorene
Aufgabenliste (1.9).

### 1.1 Pfade

In [32]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    """Walk up from the working directory until harness and system prompt are found."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_harness = (candidate / "src" / "instantrag_qwen_demo.py").is_file()
        has_prompt = (candidate / "config" / "instantrag.txt").is_file()
        if has_harness and has_prompt:
            return candidate
    raise FileNotFoundError(f"repository root not found above {start}")


ROOT = find_repo_root()
SRC_DIR = ROOT / "src"
SETTINGS_PATH = ROOT / "config" / "llm_config.json"
RESULTS_DIR = ROOT / "results"
RUNS_PATH = RESULTS_DIR / "runs.jsonl"          # experiment data, append-only
SELFTEST_PATH = RESULTS_DIR / "selftest.jsonl"  # offline self-test only
PLAYGROUND_PATH = RESULTS_DIR / "playground.jsonl"  # section 8, exploratory runs only
print("repo root:", ROOT.name)

repo root: instantrag-poc


### 1.2 Import des Harness

In [33]:
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import instantrag_qwen_demo as harness
from llm_request import FatalConfigError

assert Path(harness.__file__).resolve().parent == SRC_DIR, "harness imported from an unexpected location"
print("python :", sys.version.split()[0])
print("harness:", Path(harness.__file__).relative_to(ROOT).as_posix())

python : 3.11.18
harness: src/instantrag_qwen_demo.py


### 1.3 Konfiguration (ohne Key)

`harness.load_model()` legt den API-Key in `model_cfg["token"]` ab. Deshalb wird `model_cfg` in ein `dict` mit
geschwärzter Darstellung verpackt; angezeigt wird nur eine Whitelist. `temperature` und `max_tokens` aus
`config/llm_config.json` ignoriert der Harness – maßgeblich sind die Werte hier (Plan §2, §7).

In [34]:
REFERENCE_MODEL = "qwen-3.5-35b"  # M1 (plan O3)
TEMPERATURE = 0.1
MAX_TOKENS = 2048
PUBLIC_CFG_KEYS = ("model", "endpoint")


class SafeCfg(dict):
    """model_cfg as the harness expects it; its repr never shows the content (API key)."""

    def __repr__(self):
        return f"<SafeCfg model={self.get('model')!r} endpoint={self.get('endpoint')!r} token=***>"

    __str__ = __repr__

    def _repr_pretty_(self, printer, cycle):
        printer.text(repr(self))


def load_cfg(model=None):
    """Load the model config through the unmodified harness; optionally set the model."""
    cfg = SafeCfg(harness.load_model(SETTINGS_PATH))
    if model:
        cfg["model"] = model
    return cfg


def public_cfg(cfg):
    """The only part of the config that may be displayed or logged."""
    return {key: cfg.get(key) for key in PUBLIC_CFG_KEYS}


try:
    model_cfg = load_cfg(REFERENCE_MODEL)
    print(public_cfg(model_cfg))
except FatalConfigError as error:
    # no key on this machine: offline cells still work, live runs do not
    model_cfg = None
    print("no API key available:", error)

{'model': 'qwen-3.5-35b', 'endpoint': 'https://llm.thalabus.ai/v1/chat/completions'}


### 1.4 Metadaten und Prompt-Versionen

Die Prompt-Prüfsumme wird über den **geladenen Text** gebildet (wie `load_system_prompt()` ihn liefert), nicht
über die Dateibytes – Zeilenenden können sich zwischen Rechnern unterscheiden. Dasselbe gilt für
`harness_sha256`. `git_dirty` meldet geänderte *getrackte* Dateien, `git_dirty_files` listet sie. `file` ist die Prüfsumme der
Datei (erste 16 Zeichen) wie im Register `doc/prompt-versions.md`.

In [35]:
import hashlib
import subprocess


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _git(*args):
    try:
        done = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True)
        # rstrip only: porcelain lines start with a status column that may be a space
        return done.stdout.rstrip()
    except (OSError, subprocess.CalledProcessError):
        return None


def git_commit():
    return _git("rev-parse", "HEAD")


def git_dirty_files():
    """Tracked files that differ from the commit (sorted); None if git is unavailable."""
    status = _git("status", "--porcelain", "--untracked-files=no")
    if status is None:
        return None
    # porcelain v1: two status letters, a space, then the path ("old -> new" for renames)
    return sorted(line[3:] for line in status.splitlines() if line.strip())


def git_dirty():
    """True if tracked files differ from the commit; None if git is unavailable."""
    files = git_dirty_files()
    return None if files is None else bool(files)


def harness_sha256():
    return sha256_text(Path(harness.__file__).read_text(encoding="utf-8"))


PROMPT_PATHS = {  # registry with differences and file hashes: doc/prompt-versions.md
    "S0": ROOT / "config" / "instantrag.txt",     # Luca's original (reference)
    "S1": ROOT / "config" / "instantrag_s1.txt",  # S0 without contradictions
    "S2": ROOT / "config" / "instantrag_s2.txt",  # S1 + first-step rule + channel rule
    "S3": ROOT / "config" / "instantrag_s3.txt",  # S2 bundle: order rule, final-answer rule, "System:"
    "S3.1": ROOT / "config" / "instantrag_s3_1.txt",  # S3 iteration 1: "system" wording, example D
}


def load_prompt(version):
    path = PROMPT_PATHS[version]
    text = harness.load_system_prompt(path)
    return {
        "version": version,
        "path": path.relative_to(ROOT).as_posix(),
        "text": text,
        "sha256": sha256_text(text),
        "chars": len(text),
    }


print("git commit    :", git_commit(), "| tracked files modified:", git_dirty_files())
print("harness sha256:", harness_sha256())
for version in PROMPT_PATHS:
    prompt = load_prompt(version)
    file_sha = hashlib.sha256(PROMPT_PATHS[version].read_bytes()).hexdigest()[:16]
    print(f"prompt {version}     : {prompt['path']} | {prompt['chars']} chars | sha256 {prompt['sha256']}"
          f" | file {file_sha}")

git commit    : 626a1152714c4c3100092109506f96f0a52ff1e6 | tracked files modified: ['results/show_one_run.txt']
harness sha256: 80b18b4030351b3fe9aeb3a6a0719b46d12c84d1af67ce701e4242498ef45d98
prompt S0     : config/instantrag.txt | 2646 chars | sha256 28e20cf1b69ee3b0dcff8a48751400867db31f7a3c1895a21cca4352f8c7d1a5 | file 816d7e628a968878
prompt S1     : config/instantrag_s1.txt | 2548 chars | sha256 3d582377b503321b155fc0c440cc3b065a122766ad8051b2f1253605d5f674e3 | file 874b9777014813b6
prompt S2     : config/instantrag_s2.txt | 2841 chars | sha256 5292166a5d617b0424800d5dd61cb94060800fbd5581f69f5a87c5874fa04d8f | file 2233845d62649d96
prompt S3     : config/instantrag_s3.txt | 3037 chars | sha256 7335f50a76cc6e6b698ac954164df91e38a4e2efaba0fa4c1c2110943a574766 | file 25b1074a77b4e168
prompt S3.1     : config/instantrag_s3_1.txt | 3414 chars | sha256 a952e192ab7bf673810578ec9167dbc9cdc84e8e61f054d267747413f015187c | file 4df43ccc0e2fb9cd


**Volltext S0** (`config/instantrag.txt`, unverändert):

In [ ]:
print(load_prompt("S0")["text"])

**Volltexte S1, S2, S3 und S3.1** (`config/instantrag_s1.txt`, `config/instantrag_s2.txt`, `config/instantrag_s3.txt`, `config/instantrag_s3_1.txt`; Änderungen gegenüber S0, S1, S2 bzw. S3 in `doc/prompt-versions.md`):

In [ ]:
for version in ("S1", "S2", "S3", "S3.1"):
    prompt = load_prompt(version)
    print(f"===== {version} · {prompt['path']} · {prompt['chars']} chars · sha256 {prompt['sha256'][:16]} =====")
    print(prompt["text"])
    print()

### 1.5 JSONL-Writer (append-only)

Eine JSON-Zeile pro Lauf, mit `schema_version`. Es wird nur angehängt, nie überschrieben oder gefiltert.
Enthält eine Zeile den API-Key, wird nicht geschrieben.

In [36]:
import json

SCHEMA_VERSION = 1


def append_jsonl(path, record, secrets=()):
    """Append one record as one line. Refuses to write if a secret appears in the line."""
    line = json.dumps(record, ensure_ascii=False, default=str)
    for secret in secrets:
        if secret and secret in line:
            raise ValueError("refusing to write: the record contains a secret")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as handle:
        handle.write(line + "\n")


def read_jsonl(path):
    """Read all records, unfiltered. A missing file is an empty list."""
    path = Path(path)
    if not path.is_file():
        return []
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

### 1.6 Rekorder um `run_turn` und protokollierter Lauf

`run_logged()` ersetzt `harness.run_turn` nur für die Dauer eines Laufs durch einen Rekorder, der das Original
aufruft und Eingabe-Messages, Parameter, Rückgabe und Zeiten mitschreibt. `run_phase` läuft unverändert, seine
Ausgabe wird als `harness_log` gespeichert. `seed` geht über `PAYLOAD_EXTRAS` in den Request. Auch ein Lauf,
der mit einer Exception endet, wird geschrieben (Feld `error`). Das gilt auch für einen abgebrochenen Lauf
(`KeyboardInterrupt`, Kernel-Stopp): Der Datensatz wird geschrieben, danach wird der Abbruch weitergereicht.

In [37]:
import contextlib
import copy
import io
import time
import uuid
from datetime import datetime, timezone
from unittest import mock


def _utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")


def _error_info(error):
    return {"type": type(error).__name__, "message": str(error)}


def make_recorder(original_run_turn, turns):
    """Wrap run_turn: call the original and record input, parameters, result and timing."""

    def recording_run_turn(model_cfg, messages, max_tokens, temperature, stop_at_tool, dep_value=None):
        entry = {
            "stage": len(turns) + 1,
            "started_at": _utc_now(),
            "messages": copy.deepcopy(messages),
            "params": {
                "max_tokens": max_tokens,
                "temperature": temperature,
                "stop_at_tool": stop_at_tool,
                "dep_value": dep_value,
            },
            "payload_extras": copy.deepcopy(harness.PAYLOAD_EXTRAS),
            "result": None,
            "error": None,
            "duration_s": None,
        }
        turns.append(entry)
        started = time.perf_counter()
        try:
            result = original_run_turn(
                model_cfg, messages, max_tokens, temperature, stop_at_tool=stop_at_tool, dep_value=dep_value
            )
            entry["result"] = copy.deepcopy(result)
            return result
        except BaseException as error:  # also KeyboardInterrupt: the turn keeps its error
            entry["error"] = _error_info(error)
            raise
        finally:
            entry["duration_s"] = round(time.perf_counter() - started, 3)

    return recording_run_turn


def run_logged(model_cfg, prompt, question, phase, *, experiment, seed, out_path,
               max_tokens=MAX_TOKENS, temperature=TEMPERATURE, expected=None, note=None, extra=None):
    """Run one phase through the unmodified run_phase and append one record to out_path.

    extra: additional top-level fields (e.g. suite_line, suite_sha256); they never replace a standard field.
    """
    turns = []
    record = {
        "schema_version": SCHEMA_VERSION,
        "run_id": datetime.now().strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6],
        "started_at": _utc_now(),
        "experiment": experiment,
        "phase": phase,
        **public_cfg(model_cfg),
        "temperature": temperature,
        "max_tokens": max_tokens,
        "seed": seed,
        "prompt_version": prompt["version"],
        "prompt_path": prompt["path"],
        "prompt_sha256": prompt["sha256"],
        "prompt_chars": prompt["chars"],
        "git_commit": git_commit(),
        "git_dirty": git_dirty(),
        "git_dirty_files": git_dirty_files(),
        "harness_sha256": harness_sha256(),
        "question": question,
        "expected": expected,
        "note": note,
        "turns": turns,
        "harness_log": None,
        "error": None,
        "duration_s": None,
    }
    clash = set(extra or {}) & set(record)
    if clash:
        raise ValueError(f"extra fields must not replace standard fields: {sorted(clash)}")
    record.update(extra or {})
    extras = {} if seed is None else {"seed": seed}
    log = io.StringIO()
    started = time.perf_counter()
    aborted = None
    try:
        with mock.patch.object(harness, "run_turn", make_recorder(harness.run_turn, turns)), \
                mock.patch.object(harness, "PAYLOAD_EXTRAS", extras), \
                contextlib.redirect_stdout(log):
            harness.run_phase(
                model_cfg=model_cfg,
                system_prompt=prompt["text"],
                question=question,
                phase=phase,
                max_tokens=max_tokens,
                temperature=temperature,
            )
    except Exception as error:
        # infrastructure errors are data too: keep the run, report it separately
        record["error"] = _error_info(error)
    except BaseException as error:
        # an aborted run (KeyboardInterrupt, kernel stop) has sent requests: write it, then re-raise
        record["error"] = _error_info(error)
        aborted = error
    record["duration_s"] = round(time.perf_counter() - started, 3)
    record["harness_log"] = log.getvalue()
    append_jsonl(out_path, record, secrets=[model_cfg.get("token")])
    if record["error"]:
        print(f"run {record['run_id']} ended with an error: {record['error']}")
    if aborted is not None:
        raise aborted
    return record

### 1.7 Darstellung eines Laufs

Pro Stage: gesendete Nachrichten unverändert (der Systemprompt nur als Version + Prüfsumme, Volltext siehe 1.4),
dann die Ausgabe. Der Reasoning-Kanal ist als „Thinking“ beschriftet. Die Id eines abgefangenen Tags wird mit
`harness.assign_id` nachvollzogen (dieselbe Logik wie in `run_phase`).

In [38]:
import html

from IPython.display import HTML, display

PRE_STYLE = "white-space:pre-wrap;border:1px solid rgba(128,128,128,.5);padding:6px;margin:2px 0 10px 0"


def _pre(text):
    if not text:
        return "<p><i>(leer)</i></p>"
    return f'<pre style="{PRE_STYLE}">{html.escape(text)}</pre>'


def _table(rows):
    cells = "".join(
        f'<tr><th style="text-align:left">{html.escape(str(key))}</th>'
        f'<td style="text-align:left">{html.escape(str(value))}</td></tr>'
        for key, value in rows
    )
    return f"<table>{cells}</table>"


def message_label(index, message):
    role, content = message["role"], message["content"]
    if role == "system":
        return "Systemprompt (system)"
    if role == "assistant":
        return "Resume-Präfix (assistant, partiell)"
    if content.startswith("<RESULTS>"):
        return "RESULTS-Nachricht (user; fachlich Server-Nachricht)"
    if index == 1:
        return "Frage (user)"
    return "Korrektur-Prompt (user)"


def published_ids(record):
    """Id the harness assigned to each stage's intercepted call (None if nothing was published)."""
    counters, ids = {}, []
    for turn in record["turns"]:
        interrupt = (turn.get("result") or {}).get("interrupt")
        if interrupt and interrupt.get("value") is not None:
            ids.append(harness.assign_id(counters, interrupt["tool"]["name"]))
        else:
            ids.append(None)
    return ids


def render_run(record):
    commit = str(record["git_commit"])
    if record["git_dirty"]:
        files = record.get("git_dirty_files")
        commit += " (getrackte Dateien geändert" + (": " + ", ".join(files) if files else "") + ")"
    head = [
        ("Zeit (UTC)", record["started_at"]),
        ("Experiment", record["experiment"]),
        ("Phase", f"−{record['phase']}"),
        ("Modell", record["model"]),
        ("Endpoint", record["endpoint"]),
        ("Temperatur", record["temperature"]),
        ("max_tokens", record["max_tokens"]),
        ("seed", record["seed"]),
        ("Prompt", f"{record['prompt_version']} · {record['prompt_chars']} Zeichen · sha256 {record['prompt_sha256']}"),
        ("Git-Commit", commit),
        ("Harness sha256", record["harness_sha256"]),
        ("Aufgabe", record["question"]),
        ("Erwartet", json.dumps(record["expected"], ensure_ascii=False)),
        ("Notiz", record["note"]),
        ("Dauer (s)", record["duration_s"]),
        ("Fehler", record["error"]),
    ]
    parts = [f"<h3>Lauf {html.escape(record['run_id'])}</h3>", _table(head)]
    for turn, tag_id in zip(record["turns"], published_ids(record)):
        params = turn["params"]
        parts.append(f"<h4>Stage {turn['stage']}</h4>")
        parts.append(
            f"<p><b>Eingabe</b> (stop_at_tool={params['stop_at_tool']}, dep_value={params['dep_value']}, "
            f"payload_extras={html.escape(json.dumps(turn['payload_extras']))})</p>"
        )
        for index, message in enumerate(turn["messages"]):
            label = html.escape(message_label(index, message))
            is_logged_prompt = message["role"] == "system" and sha256_text(message["content"]) == record["prompt_sha256"]
            if is_logged_prompt:
                parts.append(f"<p>{label}: <i>{html.escape(record['prompt_version'])}, Volltext siehe Abschnitt 1.4</i></p>")
            else:
                parts.append(f"<p>{label}:</p>" + _pre(message["content"]))
        parts.append("<p><b>Ausgabe</b></p>")
        result = turn["result"]
        if result is None:
            parts.append(_table([("Fehler", turn["error"]), ("Dauer (s)", turn["duration_s"])]))
            continue
        parts.append("<p>Thinking (Reasoning-Kanal, wie empfangen):</p>" + _pre(result["reasoning_all"]))
        parts.append("<p>Antwort (Antwortkanal, wie empfangen):</p>" + _pre(result["content_all"]))
        rows = []
        interrupt = result["interrupt"]
        if interrupt is None:
            rows.append(("Abgefangener Tag", "(keiner)"))
        else:
            value = interrupt["value"]
            rows.append(("Abgefangener Tag", interrupt["tool"]["raw"]))
            rows.append(("Kanal", "Antwortkanal (Leak)" if interrupt["leak"] else "Thinking"))
            rows.append(("Wert (Server)", "(nicht auflösbar)" if value is None else harness._fmt(value)))
            rows.append(("Id", tag_id))
            rows.append(("Verworfener Rest", json.dumps(result["discarded"], ensure_ascii=False)))
        rows.append(("finish_reason", result["finish_reason"]))
        rows.append(("Usage", json.dumps(result["usage"]) if result["usage"] else "(keine)"))
        rows.append(("Dauer (s)", turn["duration_s"]))
        parts.append(_table(rows))
    parts.append("<details><summary>Harness-Log (stdout von run_phase)</summary>" + _pre(record["harness_log"]) + "</details>")
    return "\n".join(parts)


def show_run(record):
    display(HTML(render_run(record)))

### 1.8 Offline-Selbsttest (kein Request, kein echter Key)

Ein Mock für `urllib.request.urlopen` spielt die in `doc/instantrag.md` Appendix A.1 dokumentierten Ausgaben
der Phase −3 als SSE-Stream ab. Der Lauf geht durch das unveränderte `run_phase` und den Rekorder und wird nach
`results/selftest.jsonl` geschrieben. Der API-Key ist für die Dauer des Tests ein Dummy.

**Synthetischer Zusatz (nicht aus A.1):** Nach dem Tag in Stage 1 und 2 hängt der Mock ein markiertes Stück an.
Es darf nie gelesen werden – damit wird geprüft, dass der Stream am Tag endet.

Der Selbsttest prüft Mitschreiben, JSONL und Darstellung. Er sagt nichts über das Verhalten des Modells aus.

In [ ]:
import os

SELFTEST_TOKEN = "selftest-dummy-token"
SELFTEST_SEED = 12345
SELFTEST_TAIL = " [MOCK-TAIL: synthetic, must never be read]"
A1_QUESTION = "Please calculate 10+(2*4)"
A1_STAGES = [
    {"reasoning": 'I must obtain <MUL x1="2" x2="4"/>', "content": "", "tail": True},
    {"reasoning": 'I must obtain <MUL id="mul-1"/> and then obtain <ADD x1="10" x2="8"/>', "content": "", "tail": True},
    {
        "reasoning": "",
        "content": "\n\nHere is your result: 10 + (2 * 4) = 18",
        "tail": False,
        "usage": {
            "completion_tokens": 53,
            "prompt_tokens": 797,
            "total_tokens": 850,
            "completion_tokens_details": {"reasoning_tokens": 0, "text_tokens": 53},
        },
    },
]
# inputs of stages 2 and 3 after system prompt and question, verbatim from Appendix A.1
A1_INPUT_TAILS = [
    [],
    [
        {"role": "user", "content": '<RESULTS><MUL id="mul-1">8</MUL></RESULTS>'},
        {"role": "assistant", "content": '<|im_start|>I must obtain <MUL id="mul-1"/>'},
    ],
    [
        {"role": "user", "content": '<RESULTS><MUL id="mul-1">8</MUL> <ADD id="add-1">18</ADD></RESULTS>'},
        {"role": "assistant", "content": '<|im_start|>I must obtain <MUL id="mul-1"/> and then obtain <ADD id="add-1"/>'},
    ],
]


def sse_lines(stage, chunk_size=5):
    """Build the SSE byte lines of one scripted stage (small chunks, like a token stream)."""

    def pieces(text):
        return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

    def event(delta, finish_reason=None):
        return {"choices": [{"index": 0, "delta": delta, "finish_reason": finish_reason}]}

    events = [event({"reasoning_content": piece}) for piece in pieces(stage["reasoning"])]
    if stage["tail"]:
        events.append(event({"reasoning_content": SELFTEST_TAIL}))
    events += [event({"content": piece}) for piece in pieces(stage["content"])]
    events.append(event({}, finish_reason="stop"))
    if stage.get("usage"):
        events.append({"choices": [], "usage": stage["usage"]})
    lines = []
    for item in events:
        lines.append(("data: " + json.dumps(item) + "\n").encode("utf-8"))
        lines.append(b"\n")
    lines.append(b"data: [DONE]\n")
    return lines


class FakeResponse:
    def __init__(self, lines):
        self.lines = lines
        self.consumed = 0

    def __enter__(self):
        return self

    def __exit__(self, *exc_info):
        return False

    def __iter__(self):
        for line in self.lines:
            self.consumed += 1
            yield line


class FakeUrlopen:
    """Stands in for urllib.request.urlopen: no network, one scripted stream per call."""

    def __init__(self, stages):
        self.stages = stages
        self.calls = []

    def __call__(self, request, timeout=None):
        index = len(self.calls)
        assert index < len(self.stages), "more requests than scripted stages"
        response = FakeResponse(sse_lines(self.stages[index]))
        self.calls.append({
            "url": request.full_url,
            "body": json.loads(request.data.decode("utf-8")),
            # only the comparison result is kept, never the header itself
            "dummy_token_used": request.get_header("Authorization") == "Bearer " + SELFTEST_TOKEN,
            "response": response,
        })
        return response


def run_selftest():
    assert SELFTEST_PATH != RUNS_PATH
    runs_size_before = RUNS_PATH.stat().st_size if RUNS_PATH.exists() else None
    lines_before = len(read_jsonl(SELFTEST_PATH))
    original_run_turn = harness.run_turn
    original_urlopen = harness.urllib.request.urlopen
    key_env = json.loads(SETTINGS_PATH.read_text(encoding="utf-8-sig")).get("api_key_env", "LLM_API_KEY")
    prompt = load_prompt("S0")
    fake = FakeUrlopen(A1_STAGES)

    with mock.patch.dict(os.environ, {key_env: SELFTEST_TOKEN}), \
            mock.patch.object(harness.urllib.request, "urlopen", fake):
        cfg = load_cfg(REFERENCE_MODEL)
        assert cfg["token"] == SELFTEST_TOKEN, "self-test must run with the dummy token"
        assert SELFTEST_TOKEN not in repr(cfg) and SELFTEST_TOKEN not in str(cfg)
        record = run_logged(
            cfg, prompt, A1_QUESTION, 3,
            experiment="selftest",
            seed=SELFTEST_SEED,
            out_path=SELFTEST_PATH,
            expected={"answer": 18, "calls": [["MUL", [2, 4]], ["ADD", [10, 8]]]},
            note="offline self-test: mocked SSE stream replaying doc/instantrag.md Appendix A.1",
        )

    turns = record["turns"]
    # patches are gone again
    assert harness.run_turn is original_run_turn
    assert harness.urllib.request.urlopen is original_urlopen
    assert harness.PAYLOAD_EXTRAS == {}
    # run completed with exactly the three scripted requests
    assert record["error"] is None, record["error"]
    assert len(fake.calls) == 3 and len(turns) == 3
    base = [{"role": "system", "content": prompt["text"]}, {"role": "user", "content": A1_QUESTION}]
    for call, turn, input_tail in zip(fake.calls, turns, A1_INPUT_TAILS):
        assert call["dummy_token_used"]
        assert call["url"].endswith("/chat/completions")
        body = call["body"]
        assert body["seed"] == SELFTEST_SEED and body["model"] == REFERENCE_MODEL
        assert body["temperature"] == TEMPERATURE and body["max_tokens"] == MAX_TOKENS and body["stream"] is True
        # recorded messages are what was sent, and what Appendix A.1 documents
        assert turn["messages"] == body["messages"] == base + input_tail
        assert turn["payload_extras"] == {"seed": SELFTEST_SEED}
        assert turn["error"] is None
    # interceptions, values and final answer as documented
    first, second, third = (turn["result"] for turn in turns)
    assert first["interrupt"]["tool"]["raw"] == '<MUL x1="2" x2="4"/>' and first["interrupt"]["value"] == 8
    assert second["interrupt"]["tool"]["raw"] == '<ADD x1="10" x2="8"/>' and second["interrupt"]["value"] == 18
    assert not first["interrupt"]["leak"] and not second["interrupt"]["leak"]
    assert [turn["params"]["stop_at_tool"] for turn in turns] == [True, True, False]
    assert third["interrupt"] is None and third["finish_reason"] == "stop"
    assert third["content_all"] == A1_STAGES[2]["content"] and third["reasoning_all"] == ""
    assert third["usage"] == A1_STAGES[2]["usage"]
    assert published_ids(record) == ["mul-1", "add-1", None]
    # the stream was stopped at the tag: the synthetic tail was never read
    for call in fake.calls[:2]:
        assert call["response"].consumed < len(call["response"].lines)
    assert fake.calls[2]["response"].consumed == len(fake.calls[2]["response"].lines)
    assert SELFTEST_TAIL not in json.dumps(record)
    # metadata
    assert record["schema_version"] == SCHEMA_VERSION
    assert record["prompt_sha256"] == sha256_text(harness.load_system_prompt(PROMPT_PATHS["S0"]))
    assert record["git_commit"] == git_commit() and record["harness_sha256"] == harness_sha256()
    assert record["git_dirty_files"] == git_dirty_files()
    # without git both are None
    assert record["git_dirty"] == (None if record["git_dirty_files"] is None else bool(record["git_dirty_files"]))
    assert "[guard] forged tool results in model output : none" in record["harness_log"]
    # JSONL: exactly one line appended, readable, identical, no token
    stored = read_jsonl(SELFTEST_PATH)
    assert len(stored) == lines_before + 1
    assert stored[-1] == json.loads(json.dumps(record))
    assert SELFTEST_TOKEN not in SELFTEST_PATH.read_text(encoding="utf-8")
    assert (RUNS_PATH.stat().st_size if RUNS_PATH.exists() else None) == runs_size_before
    # display renders and keeps tags visible
    rendered = render_run(record)
    assert html.escape('<MUL x1="2" x2="4"/>') in rendered
    assert html.escape('<|im_start|>I must obtain <MUL id="mul-1"/>') in rendered
    print(f"SELFTEST OK: run {record['run_id']}, 3 mocked requests, "
          f"{SELFTEST_PATH.relative_to(ROOT).as_posix()} now has {len(stored)} line(s)")
    return record


selftest_record = run_selftest()

**Darstellung des Selbsttest-Laufs** (Daten aus dem Mock, kein Modellverhalten):

In [ ]:
show_run(selftest_record)

### 1.9 Aufgaben (`config/poc_items.json`, eingefroren)

Die Aufgabenlisten T1 (a+(b·c)) und T2 (a·(b+c)) werden nur gelesen, nie verändert. Die Zelle prüft die Liste
(Ids und Fragen eindeutig, Ergebnisse nachgerechnet, Kette passt zur Frage) und zeigt sie. Die Bewertung
(Abschnitt 6) nimmt richtiges Ergebnis und erwartete Calls von hier.

In [39]:
import math
import re

import pandas as pd

ITEMS_PATH = ROOT / "config" / "poc_items.json"
QUESTION_RE = re.compile(r"Please calculate (\d+)([+*])\((\d+)([+*])(\d+)\)")
OPS = {"+": "ADD", "*": "MUL"}


def apply_op(op, args):
    return sum(args) if op == "ADD" else math.prod(args)


def check_items(data, items):
    """Consistency of the frozen list: unique ids/questions, results recomputed, chain matches the question."""
    questions = [item["question"] for item in items.values()]
    assert len(set(questions)) == len(questions), "duplicate question in poc_items.json"
    for item_id, item in items.items():
        calls = item["expected_calls"]
        assert len(calls) == 2, f"{item_id}: the harness supports exactly two calls"
        for call in calls:
            assert call["op"] in ("ADD", "MUL"), item_id
            assert apply_op(call["op"], call["args"]) == call["result"], f"{item_id}: wrong result {call}"
        assert calls[0]["result"] in calls[1]["args"], f"{item_id}: second call must use the first result"
        assert item["ground_truth"] == calls[-1]["result"], f"{item_id}: ground_truth"
        assert data["sets"][item["set"]]["first_op"] == calls[0]["op"], f"{item_id}: first_op"
        match = QUESTION_RE.fullmatch(item["question"])
        assert match, f"{item_id}: question format"
        a, outer, b, inner, c = match.groups()
        inner_call = (OPS[inner], sorted([int(b), int(c)]))
        inner_result = apply_op(OPS[inner], [int(b), int(c)])
        outer_call = (OPS[outer], sorted([int(a), inner_result]))
        assert [(call["op"], sorted(call["args"])) for call in calls] == [inner_call, outer_call], item_id


def load_items(path=ITEMS_PATH):
    """Read the frozen task list (read-only) and index it by id; the set name is added as 'set'."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    items = {}
    for set_name, item_set in data["sets"].items():
        for item in item_set["items"]:
            assert item["id"] not in items, f"duplicate item id {item['id']}"
            items[item["id"]] = dict(item, set=set_name)
    check_items(data, items)
    return data, items


def format_calls(calls):
    return " → ".join(f"{call['op']}({', '.join(map(str, call['args']))}) = {call['result']}" for call in calls)


ITEMS_DATA, ITEMS = load_items()
ITEMS_BY_QUESTION = {item["question"]: item for item in ITEMS.values()}
counts = {name: len(item_set["items"]) for name, item_set in ITEMS_DATA["sets"].items()}
print(f"{ITEMS_PATH.relative_to(ROOT).as_posix()}: {len(ITEMS)} Aufgaben {counts}, "
      f"eingefroren {ITEMS_DATA['frozen_at']}, Konsistenzprüfung OK")
display(pd.DataFrame(
    [
        {
            "Id": item["id"],
            "Set": item["set"],
            "Aufgabe": item["question"],
            "erwartete Calls": format_calls(item["expected_calls"]),
            "Ergebnis": item["ground_truth"],
            "Referenz": "ja" if item["reference"] else "",
        }
        for item in ITEMS.values()
    ]
).set_index("Id"))

config/poc_items.json: 20 Aufgaben {'T1': 10, 'T2': 10}, eingefroren 2026-11-15T20:45+01:00, Konsistenzprüfung OK


,Set,Aufgabe,erwartete Calls,Ergebnis,Referenz
Id,,,,,
T1-01,T1,Please calculate 10+(2*4),"MUL(2, 4) = 8 → ADD(10, 8) = 18",18,ja
T1-02,T1,Please calculate 13+(6*7),"MUL(6, 7) = 42 → ADD(13, 42) = 55",55,
T1-03,T1,Please calculate 21+(4*9),"MUL(4, 9) = 36 → ADD(21, 36) = 57",57,
T1-04,T1,Please calculate 17+(6*8),"MUL(6, 8) = 48 → ADD(17, 48) = 65",65,
T1-05,T1,Please calculate 14+(7*9),"MUL(7, 9) = 63 → ADD(14, 63) = 77",77,
T1-06,T1,Please calculate 26+(4*8),"MUL(4, 8) = 32 → ADD(26, 32) = 58",58,
T1-07,T1,Please calculate 19+(6*9),"MUL(6, 9) = 54 → ADD(19, 54) = 73",73,
T1-08,T1,Please calculate 32+(7*8),"MUL(7, 8) = 56 → ADD(32, 56) = 88",88,
T1-09,T1,Please calculate 11+(8*9),"MUL(8, 9) = 72 → ADD(11, 72) = 83",83,


## 2. V0 – Messung prüfen (a–d)

*Gliederungspunkt – Schritt 4.* (a) `/v1/models`, (b) Cache-Test, (c) Fortsetzungstest, (d) sichtbare Zeichen
vs. `completion_tokens` (Plan §7).

## 3. Phase −1: ein Lauf, vollständig dokumentiert

*Gliederungspunkt.* Stream bis zum ersten leeren Tag, Abbruch; nichts wird zurückgeschickt.

## 4. Phase −2: ein Lauf, vollständig dokumentiert

*Gliederungspunkt.* RESULTS-Nachricht, Resume-Präfix mit Id-Verweis, zweiter Tool-Aufruf.

## 5. Phase −3: Experimente

Jedes Experiment ist eine Zeile im Register (Prompt × Modell × Aufgabenset, immer Phase −3). Pro Aufgabe läuft
genau ein Lauf durch den unveränderten Harness; Rohdaten gehen nur nach `results/runs.jsonl` (append-only), mit der
Aufgaben-Id im Datensatz. Der Seed ist je Aufgabe fest und in allen Experimenten gleich (T1-01 → 101 … T2-10 → 210),
damit sich gepaarte Läufe nur im untersuchten Faktor unterscheiden. Aufgaben, die in einem Experiment schon einen
gültigen Lauf haben, werden übersprungen; nach 2 Infrastrukturfehlern (I-ERR) in Folge bricht ein Experiment ab.
Eigene Fragenlisten ohne Register-Eintrag: Testsuites in `testsuites/` (5.4, 5.5).

**E1b** (S0 auf M2) ist ein Ersatz-Vergleich, solange M1 (`qwen-3.5-35b`) nicht erreichbar ist (Plan §2, 16.11.).
E3 und E5 sind registriert, aber noch nicht freigegeben. Die Übersicht 5.0 zeigt alle Experimente und Testsuites mit Status.

### 5.0 Übersicht: Experimente und Testsuites

Eine Zeile pro Experiment (Register 5.1) und Testsuite (`testsuites/*.txt`, 5.4): Prompt, Modell, Aufgaben, Seeds,
Status, Läufe in `results/runs.jsonl` (alle, auch I-ERR) und Aufgaben mit gültigem Lauf.

Status: **gelaufen** (jede Aufgabe hat einen gültigen Lauf) · **teilweise** · **geplant** (freigegeben, noch kein gültiger
Lauf) · **zurückgehalten** (Experiment mit `active = False` oder Suite in `HELD_SUITES`) · **fehlerhaft** (Suite-Datei nicht
lesbar) · **nicht registriert** (Läufe ohne Register-Eintrag und ohne Suite-Datei).

Beim „Run All“ erscheint die Tabelle erst am Ende von 5.5 an dieser Stelle, weil Register und Suite-Funktionen weiter
unten in Abschnitt 5 definiert werden; einzeln ausgeführt zeigt die Zelle sie sofort. Nach Läufen in 5.3 oder 5.5 wird
sie aktualisiert.

In [ ]:
from collections import Counter

# 5.0: one row per experiment (register 5.1) and test suite (testsuites/*.txt, 5.4) with status and run counts.
# Run All defines the register and the suite functions later in this section, so the table is filled in at the end
# of 5.3/5.5 (refresh_overview); run on its own, this cell shows it directly.
OVERVIEW_NEEDS = ("EXPERIMENT_REGISTER", "MODELS", "item_seed", "run_has_infra_error", "load_suite", "suite_plan",
                  "HELD_SUITES", "SUITE_DIR", "SUITE_PREFIX")
OVERVIEW_OPTIONS = ("display.max_colwidth", None, "display.max_rows", None)


def _seed_range(seeds):
    seeds = sorted(seeds)
    if not seeds:
        return "–"
    return str(seeds[0]) if len(seeds) == 1 else f"{seeds[0]}–{seeds[-1]}"


def _run_status(held, done, total):
    if held:
        return "zurückgehalten"
    if total and done >= total:
        return "gelaufen"
    return "teilweise" if done else "geplant"


def overview_table():
    """Experiments and test suites: prompt, model, tasks, seeds, status, runs (all) and tasks with a valid run."""
    records = read_jsonl(RUNS_PATH)
    runs = Counter(record.get("experiment") for record in records)
    rows = []
    for name, spec in EXPERIMENT_REGISTER.items():
        items = [item for item in ITEMS.values() if item["set"] == spec["item_set"]]
        valid = {r["expected"].get("item_id") for r in records
                 if r.get("experiment") == name and isinstance(r.get("expected"), dict) and not run_has_infra_error(r)}
        done = sum(item["id"] in valid for item in items)
        rows.append({"Name": name, "Art": "Experiment", "Prompt": spec["prompt"], "Modell": MODELS[spec["model"]],
                     "Aufgaben": f"{spec['item_set']} ({len(items)})",
                     "Seeds": _seed_range(item_seed(item["id"]) for item in items),
                     "Status": _run_status(not spec["active"], done, len(items)),
                     "Läufe": runs.get(name, 0), "gültig": f"{done}/{len(items)}",
                     "Hinweis": spec["note"] if spec["active"] else f"nicht freigegeben – {spec['note']}"})
    for path in sorted(SUITE_DIR.glob("*.txt")):
        name, experiment = path.name, SUITE_PREFIX + path.name
        try:
            suite = load_suite(name)
        except (ValueError, UnicodeDecodeError) as error:
            rows.append({"Name": experiment, "Art": "Testsuite", "Prompt": "–", "Modell": "–", "Aufgaben": "–",
                         "Seeds": "–", "Status": "fehlerhaft", "Läufe": runs.get(experiment, 0), "gültig": "–",
                         "Hinweis": " ".join(str(error).split())})
            continue
        plan, warnings = suite_plan(suite)
        done = sum(not row["pending"] for row in plan)
        ids = [entry["item"]["id"] for entry in suite["entries"]]
        held = [HELD_SUITES[name]] if name in HELD_SUITES else []
        rows.append({"Name": experiment, "Art": "Testsuite", "Prompt": suite["prompt"], "Modell": suite["model"],
                     "Aufgaben": f"{ids[0]} … {ids[-1]} ({len(ids)})",
                     "Seeds": _seed_range(entry["seed"] for entry in suite["entries"]),
                     "Status": _run_status(name in HELD_SUITES, done, len(ids)),
                     "Läufe": runs.get(experiment, 0), "gültig": f"{done}/{len(ids)}",
                     "Hinweis": "; ".join(held + warnings)})
    listed = {row["Name"] for row in rows}
    for name in sorted(n for n in runs if n not in listed):  # runs without register entry or suite file
        rows.append({"Name": name, "Art": "nur Daten", "Prompt": "–", "Modell": "–", "Aufgaben": "–", "Seeds": "–",
                     "Status": "nicht registriert", "Läufe": runs[name], "gültig": "–",
                     "Hinweis": "kein Register-Eintrag und keine Suite-Datei"})
    return pd.DataFrame(rows).set_index("Name")


def _overview_content():
    if any(name not in globals() for name in OVERVIEW_NEEDS):
        return HTML("<p><i>Die Übersicht folgt: Beim „Run All“ wird sie am Ende von 5.5 hier eingesetzt (Register 5.1 "
                    "und Testsuites 5.4 sind dann definiert). Einzeln ausgeführt zeigt diese Zelle sie sofort.</i></p>")
    return overview_table()


def refresh_overview():
    """Redraw the 5.0 table in place (called at the end of 5.3 and 5.5); without a display handle nothing happens."""
    handle = globals().get("OVERVIEW_HANDLE")
    if handle is not None:
        with pd.option_context(*OVERVIEW_OPTIONS):
            handle.update(_overview_content())


with pd.option_context(*OVERVIEW_OPTIONS):
    OVERVIEW_HANDLE = display(_overview_content(), display_id=True)

### 5.1 Register, Vorschau und Schleife

In [40]:
from collections import Counter

EXPERIMENT_PHASE = 3
MAX_IERR_IN_A_ROW = 2
MODELS = {"M1": "qwen-3.5-35b", "M2": "qwen-3.8-180b-next"}
EXPERIMENT_REGISTER = {
    "E1": {"prompt": "S0", "model": "M1", "item_set": "T1", "active": True, "note": "baseline"},
    "E1b": {"prompt": "S0", "model": "M2", "item_set": "T1", "active": True,
            "note": "substitute baseline on M2 while M1 is unreachable (plan §2, 2026-11-16)"},
    "E2": {"prompt": "S1", "model": "M1", "item_set": "T1", "active": True, "note": "prompt S0 -> S1"},
    "E4": {"prompt": "S1", "model": "M2", "item_set": "T1", "active": True, "note": "model M1 -> M2"},
    "E3": {"prompt": "S2", "model": "M1", "item_set": "T1", "active": False, "note": "prompt S1 -> S2"},
    "E5": {"prompt": "S1", "model": "M1", "item_set": "T2", "active": False, "note": "tasks T1 -> T2"},
    "E7": {"prompt": "S3", "model": "M2", "item_set": "T1", "active": True,
           "note": "bundled prompt S3 on M2, compared with E4 (S1) and E1b (S0) (plan §2, 2026-11-16)"},
    "E7.1": {"prompt": "S3.1", "model": "M2", "item_set": "T1", "active": True,
             "note": "S3 iteration 1 on the development set T1, compared with E7 (plan §2, 2026-11-16)"},
    "E9": {"prompt": "S3.1", "model": "M1", "item_set": "T1", "active": True,
           "note": "frozen S3.1 on M1, compared with E1 (S0) and E7.1 (M2) (plan §2, 2026-11-18)"},
    "VAL-T2": {"prompt": "S3.1", "model": "M2", "item_set": "T2", "active": False,
               "note": "validation of the frozen S3.1 on T2, held until Roland releases it (plan §2, 2026-11-17)"},
}


def item_seed(item_id):
    """Fixed per task, identical in every experiment: T1-01 -> 101, ..., T2-10 -> 210."""
    return int(item_id[1]) * 100 + int(item_id.split("-")[1])


def run_has_infra_error(record):
    """I-ERR exactly as in 6.3, from the raw record alone (this section runs before section 6 is defined)."""
    if record.get("error") or not record["turns"]:
        return True
    for turn in record["turns"]:
        result = turn.get("result")
        if turn.get("error") or result is None:
            return True
        if result["interrupt"] is None and result.get("finish_reason") is None:
            return True
    return False


def experiment_plan(experiments, out_path=None):
    """One row per experiment and task; a task with a valid run in out_path is not pending."""
    existing = read_jsonl(Path(out_path or RUNS_PATH))
    rows = []
    for name in dict.fromkeys(experiments):
        spec = EXPERIMENT_REGISTER.get(name)
        if spec is None:
            raise ValueError(f"unknown experiment {name!r}; registered: {sorted(EXPERIMENT_REGISTER)}")
        if not spec["active"]:
            raise ValueError(f"experiment {name} is registered but not released yet")
        mine = [r for r in existing if r.get("experiment") == name and isinstance(r.get("expected"), dict)]
        valid = {r["expected"].get("item_id") for r in mine if not run_has_infra_error(r)}
        failed = Counter(r["expected"].get("item_id") for r in mine if run_has_infra_error(r))
        for item in ITEMS.values():
            if item["set"] != spec["item_set"]:
                continue
            if item["id"] in valid:
                status = "gültiger Lauf vorhanden – übersprungen"
            else:
                status = "offen" + (f" (bisher {failed[item['id']]}× I-ERR)" if failed[item["id"]] else "")
            rows.append({"experiment": name, "item_id": item["id"], "question": item["question"],
                         "prompt": spec["prompt"], "model": MODELS[spec["model"]], "seed": item_seed(item["id"]),
                         "pending": item["id"] not in valid, "status": status})
    return rows


def run_experiments(experiments, run, cfg=None, out_path=None, show=True):
    """Preview the plan; only with run=True send the pending runs via run_logged and return their records."""
    out_path = Path(out_path or RUNS_PATH)
    if out_path.resolve() in (PLAYGROUND_PATH.resolve(), SELFTEST_PATH.resolve()):
        raise ValueError("experiments never write to the playground or self-test file")
    rows = experiment_plan(experiments, out_path)
    pending = [row for row in rows if row["pending"]]
    target = out_path.relative_to(ROOT).as_posix() if out_path.is_relative_to(ROOT) else str(out_path)
    print(f"experiments {list(dict.fromkeys(experiments))}: {len(pending)} of {len(rows)} runs pending "
          f"(phase −{EXPERIMENT_PHASE}, temperature {TEMPERATURE}, max_tokens {MAX_TOKENS}) → {target}"
          + ("" if run else " | RUN_EXPERIMENTE = False: nothing is sent"))
    if show and rows:
        table = pd.DataFrame(rows).drop(columns="pending").rename(columns={
            "experiment": "Experiment", "item_id": "Aufgabe", "question": "Frage", "prompt": "Prompt",
            "model": "Modell", "seed": "Seed", "status": "Status"})
        with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
            display(table)
    if not run:
        return []
    cfg = model_cfg if cfg is None else cfg
    if cfg is None:
        raise RuntimeError("no API key loaded (section 1.3): a live run is not possible")
    records, stopped, in_a_row = [], set(), Counter()
    for number, row in enumerate(pending, 1):
        name, item = row["experiment"], ITEMS[row["item_id"]]
        if name in stopped:
            continue
        run_cfg = SafeCfg(cfg)
        run_cfg["model"] = row["model"]
        record = run_logged(
            run_cfg, load_prompt(row["prompt"]), item["question"], EXPERIMENT_PHASE,
            experiment=name, seed=row["seed"], out_path=out_path,
            expected={"item_id": item["id"], "answer": item["ground_truth"],
                      "calls": [[call["op"], call["args"]] for call in item["expected_calls"]]},
            note=f"{name}: {EXPERIMENT_REGISTER[name]['note']}",
        )
        records.append(record)
        failed = run_has_infra_error(record)
        in_a_row[name] = in_a_row[name] + 1 if failed else 0
        print(f"[{number}/{len(pending)}] {name} {item['id']} → {record['run_id']} · "
              f"{'I-ERR' if failed else str(len(record['turns'])) + ' requests'} · {record['duration_s']} s")
        if in_a_row[name] >= MAX_IERR_IN_A_ROW:
            stopped.add(name)
            print(f"{name} stopped: {MAX_IERR_IN_A_ROW} infrastructure errors in a row (model or gateway unreachable?)")
    errors = sum(run_has_infra_error(record) for record in records)
    print(f"done: {len(records)} runs written to {target}, {errors} with I-ERR"
          + (f"; stopped early: {sorted(stopped)}" if stopped else ""))
    return records

### 5.2 Offline-Test der Schleife (kein Request)

Der Mock aus 1.8 spielt für jede Aufgabe den A.1-Lauf ab; geschrieben wird in eine temporäre Datei, nie nach `results/runs.jsonl`. Geprüft werden Vorschau, Zuordnung (Experiment, Prompt, Modell, Aufgabe, Seed im Request), Überspringen gültiger Läufe und der Abbruch nach 2 I-ERR.

In [ ]:
import tempfile
import urllib.error


def run_experiment_loop_checks():
    """Offline: preview, loop, skipping, shared seeds, outage stop, refusals; never touches results/runs.jsonl."""
    cfg = SafeCfg({"model": "placeholder", "endpoint": "https://example.invalid/v1/chat/completions",
                   "token": SELFTEST_TOKEN})
    t1 = [item for item in ITEMS.values() if item["set"] == "T1"]
    runs_before = RUNS_PATH.stat().st_size if RUNS_PATH.exists() else None
    original_urlopen = harness.urllib.request.urlopen

    def go(experiments, run, fake, target):
        with mock.patch.object(harness.urllib.request, "urlopen", fake), contextlib.redirect_stdout(io.StringIO()):
            return run_experiments(experiments, run, cfg=cfg, out_path=target, show=False)

    def outage(request, timeout=None):
        raise urllib.error.URLError("mock: model unreachable")

    with tempfile.TemporaryDirectory() as tmp:
        target = Path(tmp) / "runs.jsonl"
        # preview only: nothing sent, nothing written
        fake = FakeUrlopen([])
        assert go(["E4"], False, fake, target) == [] and not fake.calls and not target.exists()
        plan = experiment_plan(["E4"], target)
        assert [row["item_id"] for row in plan] == [item["id"] for item in t1] and all(row["pending"] for row in plan)
        assert [row["seed"] for row in plan] == list(range(101, 111)) and item_seed("T2-10") == 210
        # E4: one run per task, the A.1 replay stands in for the model (3 requests per run)
        fake = FakeUrlopen(A1_STAGES * len(t1))
        records = go(["E4"], True, fake, target)
        assert len(records) == len(t1) == len(read_jsonl(target)) and len(fake.calls) == 3 * len(t1)
        s1 = load_prompt("S1")
        for index, (record, item) in enumerate(zip(records, t1)):
            assert (record["experiment"], record["phase"], record["model"]) == ("E4", 3, "qwen-3.8-180b-next")
            assert record["prompt_version"] == "S1" and record["prompt_sha256"] == s1["sha256"]
            assert record["question"] == item["question"] and record["expected"]["item_id"] == item["id"]
            assert record["seed"] == item_seed(item["id"]) and not run_has_infra_error(record)
            for call in fake.calls[3 * index:3 * index + 3]:
                body = call["body"]
                assert body["model"] == "qwen-3.8-180b-next" and body["seed"] == record["seed"]
                assert body["messages"][0]["content"] == s1["text"]
                assert body["messages"][1]["content"] == item["question"]
        assert cfg["model"] == "placeholder", "the caller's cfg must not be modified"
        # a second pass finds every task done
        fake = FakeUrlopen([])
        assert go(["E4"], True, fake, target) == [] and not fake.calls and len(read_jsonl(target)) == len(t1)
        assert not any(row["pending"] for row in experiment_plan(["E4"], target))
        # E1b: other prompt, same model, same seeds per task
        records_b = go(["E1b"], True, FakeUrlopen(A1_STAGES * len(t1)), target)
        assert [(r["prompt_version"], r["model"]) for r in records_b] == [("S0", "qwen-3.8-180b-next")] * len(t1)
        assert [r["seed"] for r in records_b] == [r["seed"] for r in records]
        # outage: E1 stops after two I-ERR, the failed tasks stay pending; E4 has nothing left
        failed = go(["E1", "E4"], True, outage, target)
        assert len(failed) == MAX_IERR_IN_A_ROW and all(run_has_infra_error(r) for r in failed)
        assert {r["experiment"] for r in failed} == {"E1"} and len(read_jsonl(target)) == 2 * len(t1) + 2
        plan = experiment_plan(["E1"], target)
        assert all(row["pending"] for row in plan) and "1× I-ERR" in plan[0]["status"] and plan[2]["status"] == "offen"
        # refusals
        for bad in (["E99"], ["E3"], ["VAL-T2"]):  # unknown, not released, held validation
            try:
                experiment_plan(bad, target)
                raise RuntimeError(f"{bad} accepted")
            except ValueError:
                pass
        try:
            run_experiments(["E4"], False, cfg=cfg, out_path=PLAYGROUND_PATH, show=False)
            raise RuntimeError("playground file accepted")
        except ValueError:
            pass
    assert harness.urllib.request.urlopen is original_urlopen
    assert (RUNS_PATH.stat().st_size if RUNS_PATH.exists() else None) == runs_before
    return len(t1)


print(f"experiment loop checks OK: preview, {run_experiment_loop_checks()} mocked runs per experiment into a "
      "temporary file, skip of valid runs, shared seeds, stop after 2 I-ERR, refusals")

### 5.3 Experimente ausführen

Mit `RUN_EXPERIMENTE = False` zeigt die Zelle nur die Vorschau. Für einen echten Lauf `True` setzen und **nur diese Zelle** ausführen; danach wieder auf `False` stellen. Auswertung: Abschnitt 6.8, einzelne Läufe: 6.9 mit `QUELLE = "runs"`.

In [41]:
EXPERIMENTE = ["E1", "E9"]    # keys of EXPERIMENT_REGISTER, e.g. ["E4", "E1b"]
RUN_EXPERIMENTE = False    # True sends requests to the LLM endpoint and appends to results/runs.jsonl

_missing = [name for name in ("run_logged", "ITEMS", "run_experiments") if name not in globals()]
if _missing:
    raise RuntimeError("Einrichtung fehlt: Abschnitt 1 (1.1–1.9, ohne 1.8) und Zelle 5.1 ausführen.")
experiment_records = run_experiments(EXPERIMENTE, RUN_EXPERIMENTE)
if "refresh_overview" in globals():
    refresh_overview()  # table in 5.0 with the current run counts

experiments ['E7.1']: 10 of 10 runs pending (phase −3, temperature 0.1, max_tokens 2048) → results/runs.jsonl


,Experiment,Aufgabe,Frage,Prompt,Modell,Seed,Status
0,E7.1,T1-01,Please calculate 10+(2*4),S3.1,qwen-3.8-180b-next,101,offen
1,E7.1,T1-02,Please calculate 13+(6*7),S3.1,qwen-3.8-180b-next,102,offen
2,E7.1,T1-03,Please calculate 21+(4*9),S3.1,qwen-3.8-180b-next,103,offen
3,E7.1,T1-04,Please calculate 17+(6*8),S3.1,qwen-3.8-180b-next,104,offen
4,E7.1,T1-05,Please calculate 14+(7*9),S3.1,qwen-3.8-180b-next,105,offen
5,E7.1,T1-06,Please calculate 26+(4*8),S3.1,qwen-3.8-180b-next,106,offen
6,E7.1,T1-07,Please calculate 19+(6*9),S3.1,qwen-3.8-180b-next,107,offen
7,E7.1,T1-08,Please calculate 32+(7*8),S3.1,qwen-3.8-180b-next,108,offen
8,E7.1,T1-09,Please calculate 11+(8*9),S3.1,qwen-3.8-180b-next,109,offen
9,E7.1,T1-10,Please calculate 31+(4*7),S3.1,qwen-3.8-180b-next,110,offen


[1/10] E7.1 T1-01 → 20261116T211319-af67ae · 4 requests · 8.446 s
[2/10] E7.1 T1-02 → 20261116T211327-edf98a · 3 requests · 4.412 s
[3/10] E7.1 T1-03 → 20261116T211332-0e0cdc · 3 requests · 5.087 s
[4/10] E7.1 T1-04 → 20261116T211337-26b74f · 3 requests · 4.357 s
[5/10] E7.1 T1-05 → 20261116T211342-e0f05f · 4 requests · 5.915 s
[6/10] E7.1 T1-06 → 20261116T211348-0cea4c · 4 requests · 13.261 s
[7/10] E7.1 T1-07 → 20261116T211401-ff9f86 · 4 requests · 6.043 s
[8/10] E7.1 T1-08 → 20261116T211407-c91feb · 3 requests · 5.383 s
[9/10] E7.1 T1-09 → 20261116T211413-3cd79f · 4 requests · 6.487 s
[10/10] E7.1 T1-10 → 20261116T211419-91d7ec · 3 requests · 6.507 s
done: 10 runs written to results/runs.jsonl, 0 with I-ERR


### 5.4 Testsuites

Eine Testsuite ist eine Textdatei in `testsuites/` (Beispiel: `testsuites/beispiel.txt`):

```
# model: qwen-3.8-180b-next      (voller Name oder M1/M2)
# prompt: S3.1                   (eine in 1.4 registrierte Version)
# seed: 301                      (Seed der ersten Frage; Frage i bekommt Start + i, i ab 0)
Please calculate 12+(3*7)
Please calculate 6*(5+8)
```

Die drei Kopfzeilen stehen vor der ersten Frage; danach eine Frage pro Zeile, wörtlich gesendet. Leere Zeilen und
andere `#`-Zeilen werden übersprungen. Sollwert und erwartete Calls werden aus der Frage berechnet: Der Ausdruck ab der
ersten Ziffer bzw. Klammer muss genau `a op (b op c)` oder (seit 17.11.) `(b op c) op a` sein – nur Ganzzahlen,
nur `+` und `*`, genau eine Klammer (die Klammer wird zuerst berechnet). Andere Formen werden mit Zeilennummer und Grund abgelehnt; die Regeln von T1/T2 (z. B. keine
0 oder 1) gelten für Suites nicht, `config/poc_items.json` bleibt unberührt.

Läufe gehen nach `results/runs.jsonl` mit `experiment = "suite:<dateiname>"`, Aufgaben-Id `<name>-L01` … (Frage-Index),
und den Feldern `suite_line` (Zeile in der Datei) und `suite_sha256` (Prüfsumme des Dateitexts; LF und CRLF ergeben
denselben Wert). Eine Frage mit gültigem
Lauf und gleichem Text wird übersprungen. Wurde die Datei nach dem ersten Lauf geändert, warnen Vorschau und Lauf.
Bewertung (6.3), Raten (6.6), Export (6.10) und `results/show_runs.py` behandeln eine Suite wie ein Experiment.

**Zurückgehaltene Suites** (`HELD_SUITES` in der Zelle unten): Vorschau ja, Senden nein; freigegeben wird durch
Entfernen des Eintrags. Zurückgehalten sind die Validierungs-Suites `val_t1_new.txt` und `val_forms.txt` (erzeugt mit
`testsuites/make_validation_suites.py`, Plan §2 vom 17.11.); ihre Prompt-Kopfzeile S3.1 ist die eingefrorene
Version (Plan §2, 17.11.).

In [ ]:
SUITE_DIR = ROOT / "testsuites"
SUITE_PREFIX = "suite:"
SUITE_HEADER_RE = re.compile(r"#\s*(model|prompt|seed)\s*:\s*(.*?)\s*$", re.IGNORECASE)
SUITE_EXPR_RE = re.compile(r"(\d+)\s*([+*])\s*\(\s*(\d+)\s*([+*])\s*(\d+)\s*\)")
SUITE_EXPR_LEFT_RE = re.compile(r"\(\s*(\d+)\s*([+*])\s*(\d+)\s*\)\s*([+*])\s*(\d+)")  # (b op c) op a, since 17.11.
SUITE_OPS = {"+": ("ADD", lambda x, y: x + y), "*": ("MUL", lambda x, y: x * y)}


def parse_suite_question(question):
    """(a, op1, b, op2, c) for 'a op1 (b op2 c)' and '(b op2 c) op1 a'; ValueError with a clear reason otherwise.

    Both forms give the same tuple: the bracket (b op2 c) is computed first, then op1 with the other operand a.
    """
    start = re.search(r"[\d(]", question)
    if start is None:
        raise ValueError("kein Rechenausdruck (keine Zahl)")
    expr = question[start.start():].strip().rstrip("?.!").strip()
    match = SUITE_EXPR_RE.fullmatch(expr)
    if match:
        return int(match[1]), match[2], int(match[3]), match[4], int(match[5])
    match = SUITE_EXPR_LEFT_RE.fullmatch(expr)
    if match:
        return int(match[5]), match[4], int(match[1]), match[2], int(match[3])
    if re.search(r"\d\s*[.,]\s*\d", expr):
        reason = "nur Ganzzahlen erlaubt"
    elif re.search(r"[-/:xX×÷^]", expr):
        reason = "nur + und * erlaubt"
    elif expr.count("(") != 1 or expr.count(")") != 1:
        reason = "genau eine Klammer erforderlich"
    else:
        reason = "Ausdruck hat nicht die Form a op (b op c) oder (b op c) op a"
    raise ValueError(f"{reason}: {expr!r}")


def suite_item(question, item_id, item_set):
    """Item in the shape of config/poc_items.json, computed from the question (inner operation first)."""
    a, op1, b, op2, c = parse_suite_question(question)
    (inner, f_inner), (outer, f_outer) = SUITE_OPS[op2], SUITE_OPS[op1]
    r1 = f_inner(b, c)
    total = f_outer(a, r1)
    return {"id": item_id, "set": item_set, "question": question, "ground_truth": total,
            "expected_calls": [{"op": inner, "args": [b, c], "result": r1},
                               {"op": outer, "args": [a, r1], "result": total}]}


def suite_text_sha256(text):
    """SHA-256 of a suite text with unified line endings: a Windows checkout (CRLF) and LF give the same value."""
    return hashlib.sha256(text.replace("\r\n", "\n").encode("utf-8")).hexdigest()


def load_suite(name):
    """Parse testsuites/<name>; ValueError listing every problem (line number and reason)."""
    path = SUITE_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Testsuite {path.relative_to(ROOT).as_posix()} nicht gefunden")
    content = path.read_bytes().decode("utf-8-sig")
    header, entries, problems = {}, [], []
    stem, experiment = Path(name).stem, SUITE_PREFIX + name
    for line_no, line in enumerate(content.splitlines(), 1):
        text = line.strip()
        if not text:
            continue
        if text.startswith("#"):
            match = SUITE_HEADER_RE.fullmatch(text)
            if match:
                key = match[1].lower()
                if entries:
                    problems.append(f"Zeile {line_no}: Kopfzeile '# {key}:' nach der ersten Frage")
                elif key in header:
                    problems.append(f"Zeile {line_no}: '# {key}:' doppelt")
                header[key] = match[2]
            continue
        try:
            item = suite_item(text, f"{stem}-L{len(entries) + 1:02d}", experiment)
        except ValueError as error:
            problems.append(f"Zeile {line_no}: {error}")
            continue
        entries.append({"index": len(entries), "line": line_no, "item": item})
    for key in ("model", "prompt", "seed"):
        if key not in header:
            problems.append(f"Kopfzeile '# {key}: …' fehlt")
    model = MODELS.get(header.get("model"), header.get("model"))
    if "model" in header and model not in MODELS.values():
        problems.append(f"unbekanntes Modell {header['model']!r}; erlaubt: {sorted(MODELS) + sorted(MODELS.values())}")
    if "prompt" in header and header["prompt"] not in PROMPT_PATHS:
        problems.append(f"unbekannte Prompt-Version {header['prompt']!r}; erlaubt: {sorted(PROMPT_PATHS)}")
    if "seed" in header and not re.fullmatch(r"\d+", header["seed"]):
        problems.append(f"Seed muss eine Ganzzahl sein, nicht {header['seed']!r}")
    if not entries and not problems:
        problems.append("keine Frage in der Datei")
    if problems:
        raise ValueError(f"Testsuite {name}:\n  " + "\n  ".join(problems))
    seed = int(header["seed"])
    for entry in entries:
        entry["seed"] = seed + entry["index"]
    return {"name": name, "experiment": experiment, "path": path, "sha256": suite_text_sha256(content),
            "model": model, "prompt": header["prompt"], "seed_start": seed, "entries": entries}


def suite_plan(suite, out_path=None):
    """One row per question; skip a question that has a valid run with the same text; warn about a changed file."""
    mine = [r for r in read_jsonl(Path(out_path or RUNS_PATH)) if r.get("experiment") == suite["experiment"]]
    valid = {(r["expected"].get("item_id"), r["question"]) for r in mine
             if isinstance(r.get("expected"), dict) and not run_has_infra_error(r)}
    warnings = []
    old_hashes = sorted({r.get("suite_sha256") for r in mine} - {suite["sha256"]})
    if old_hashes:
        earlier = {r["expected"].get("item_id"): r["question"] for r in mine if isinstance(r.get("expected"), dict)}
        changed = [e["item"]["id"] for e in suite["entries"]
                   if e["item"]["id"] in earlier and earlier[e["item"]["id"]] != e["item"]["question"]]
        warnings.append(f"WARNUNG: {suite['name']} wurde nach dem ersten Lauf geändert (frühere Prüfsumme "
                        f"{', '.join(h[:16] for h in old_hashes if h)}, jetzt {suite['sha256'][:16]})"
                        + (f"; andere Frage als früher: {', '.join(changed)}" if changed else ""))
    rows = []
    for entry in suite["entries"]:
        item = entry["item"]
        done = (item["id"], item["question"]) in valid
        rows.append({"experiment": suite["experiment"], "item_id": item["id"], "line": entry["line"],
                     "question": item["question"], "seed": entry["seed"], "ground_truth": item["ground_truth"],
                     "calls": format_calls(item["expected_calls"]), "prompt": suite["prompt"], "model": suite["model"],
                     "pending": not done, "status": "gültiger Lauf vorhanden – übersprungen" if done else "offen"})
    return rows, warnings


# held suites (plan §2, 17.11.): preview only; run_suite refuses to send until the entry is removed here
HELD_SUITES = {
    "val_t1_new.txt": "Validierung T1-Form mit neuen Zahlen (S3.1): wartet auf Rolands Freigabe",
    "val_forms.txt": "Validierung andere Formen (S3.1): wartet auf Rolands Freigabe",
}


def run_suite(name, run, cfg=None, out_path=None, show=True):
    """Preview a test suite; only with run=True send the pending questions via run_logged (phase −3)."""
    out_path = Path(out_path or RUNS_PATH)
    if out_path.resolve() in (PLAYGROUND_PATH.resolve(), SELFTEST_PATH.resolve()):
        raise ValueError("suites never write to the playground or self-test file")
    suite = load_suite(name)
    rows, warnings = suite_plan(suite, out_path)
    pending = [row for row in rows if row["pending"]]
    target = out_path.relative_to(ROOT).as_posix() if out_path.is_relative_to(ROOT) else str(out_path)
    for warning in warnings:
        print(warning)
    print(f"{suite['experiment']} · {suite['prompt']} · {suite['model']} · Seed ab {suite['seed_start']} · "
          f"sha256 {suite['sha256'][:16]}: {len(pending)} of {len(rows)} runs pending (phase −{EXPERIMENT_PHASE}) → "
          f"{target}" + ("" if run else " | RUN_SUITE = False: nothing is sent"))
    if name in HELD_SUITES:
        print(f"zurückgehalten: {HELD_SUITES[name]} – Senden gesperrt (Freigabe: Eintrag aus HELD_SUITES entfernen)")
    if show:
        table = pd.DataFrame(rows).drop(columns=["pending", "experiment", "prompt", "model"]).rename(columns={
            "item_id": "Aufgabe", "line": "Zeile", "question": "Frage", "seed": "Seed",
            "ground_truth": "Sollwert", "calls": "erwartete Calls", "status": "Status"})
        with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
            display(table)
    if not run:
        return []
    if name in HELD_SUITES:
        raise ValueError(f"Testsuite {name} ist zurückgehalten ({HELD_SUITES[name]}); zum Freigeben den Eintrag "
                         "aus HELD_SUITES (Zelle 5.4) entfernen")
    cfg = model_cfg if cfg is None else cfg
    if cfg is None:
        raise RuntimeError("no API key loaded (section 1.3): a live run is not possible")
    prompt = load_prompt(suite["prompt"])
    by_id = {entry["item"]["id"]: entry for entry in suite["entries"]}
    records, in_a_row = [], 0
    for number, row in enumerate(pending, 1):
        entry = by_id[row["item_id"]]
        item = entry["item"]
        run_cfg = SafeCfg(cfg)
        run_cfg["model"] = suite["model"]
        record = run_logged(
            run_cfg, prompt, item["question"], EXPERIMENT_PHASE, experiment=suite["experiment"], seed=entry["seed"],
            out_path=out_path,
            expected={"item_id": item["id"], "answer": item["ground_truth"],
                      "calls": [[call["op"], call["args"]] for call in item["expected_calls"]]},
            note=f"test suite {suite['name']}, line {entry['line']}",
            extra={"suite_line": entry["line"], "suite_sha256": suite["sha256"]},
        )
        records.append(record)
        failed = run_has_infra_error(record)
        in_a_row = in_a_row + 1 if failed else 0
        print(f"[{number}/{len(pending)}] {item['id']} → {record['run_id']} · "
              f"{'I-ERR' if failed else str(len(record['turns'])) + ' requests'} · {record['duration_s']} s")
        if in_a_row >= MAX_IERR_IN_A_ROW:
            print(f"stopped: {MAX_IERR_IN_A_ROW} infrastructure errors in a row (model or gateway unreachable?)")
            break
    errors = sum(run_has_infra_error(record) for record in records)
    print(f"done: {len(records)} runs written to {target}, {errors} with I-ERR")
    return records

### 5.5 Testsuite ausführen

Mit `RUN_SUITE = False` zeigt die Zelle nur die Vorschau (Zeile, Aufgabe, Frage, Seed, Sollwert, erwartete Calls, Status). Für einen echten Lauf `True` setzen und **nur diese Zelle** ausführen; danach wieder auf `False` stellen. Auswertung: 6.8 und Export 6.10.

In [ ]:
SUITE = "beispiel.txt"   # file in testsuites/
RUN_SUITE = False        # True sends requests to the LLM endpoint and appends to results/runs.jsonl

_missing = [name for name in ("run_logged", "run_suite", "MODELS") if name not in globals()]
if _missing:
    raise RuntimeError("Einrichtung fehlt: Abschnitt 1 (1.1–1.9, ohne 1.8), Zelle 5.1 und 5.4 ausführen.")
suite_records = run_suite(SUITE, RUN_SUITE)
if "refresh_overview" in globals():
    refresh_overview()  # table in 5.0 with the current run counts

## 6. Auswertung

Bewertet wird aus den Rohdaten (gesendete Messages und Rückgabe von `run_turn` je Stage), nicht aus dem Urteil des
Harness. Sein Guard-Urteil, „quoted in answer“ und `[halt]` stehen nur daneben (Plan §9, Prinzip 3). Sollwerte
kommen aus `config/poc_items.json` (1.9).

- **6.1** Regeln: Codes, Erfolgsstufen, Auslegung · **6.2–6.4** Bewertung · **6.5–6.6** Darstellung und Statistik
- **6.7** Offline-Tests der Bewertung (synthetische Läufe, kein Request)
- **6.8** Auswertung von `results/runs.jsonl` und `results/playground.jsonl`, getrennt beschriftet

### 6.1 Regeln der Bewertung (freigegeben 15.11.2026)

Grundlage ist Plan §8. Zahlen werden als ganze Zahl verglichen (`(?<!\d)N(?!\d)`; auch `18.5` und `0.18` zählen
nicht als 18), nie als Teilstring. Die Argumentreihenfolge in einem Call ist egal, die Reihenfolge der Operationen
fest. Bewertet wird je Stage (ein Request) die Ausgabe bis zum Stopp. Ein Lauf hat die Schritte 1 und 2 (Calls)
und 3 (Endantwort, nur Phase −3).

| Code | gesetzt, wenn … | Wirkung |
|---|---|---|
| A-OMIT | in einer Stage mit Tool-Stopp kein Tag versucht wurde | Fehler |
| A-MALF | ein Tag versucht wurde, den der Harness nicht ausführt (einfache Anführungszeichen, Kleinschreibung, `<MUL …></MUL>`, offenes Tag, Bindestrich-Attribut) | Fehler, auch wenn danach ein gültiger Tag folgt |
| A-WRONG | der ausgeführte Call falsche Operation oder Argumente hat (Schritt 2 gemessen am tatsächlich veröffentlichten Wert), `<DEP>` ohne Serverwert, ein Id-Verweis auf ein nicht veröffentlichtes Ergebnis, oder nach beiden Ergebnissen noch ein Call steht | Fehler |
| A-LEAK | ein Call (auch fehlerhaft) im Antwortkanal steht | bricht nur strict |
| A-META | der Harness einen Platzhalter-Tag abfängt (`x1="..."`) | Fehler |
| B-PREM | ein noch nicht veröffentlichtes Ergebnis im Antwortkanal steht (Zahlen aus der Frage ausgenommen) | Fehler, zählt auch bei lenient |
| B-FORGE | ein geschlossener Tag mit Wert kein exaktes Echo ist, in beiden Kanälen, Bindestrich-Attribute eingeschlossen | Fehler, zählt auch bei lenient |
| B-SELF | ein noch nicht veröffentlichtes Ergebnis im Thinking steht | nur beschreibend |
| C-IGNORE | ein Call ab Schritt 2 den letzten Serverwert nicht verwendet | Fehler |
| C-FINAL | (Phase −3) die Endantwort fehlt, ≠ richtiges Ergebnis oder ≠ letzter Serverwert ist; auch eine vorzeitige Antwort mit Zahl | Fehler |
| R-EMPTY | kein Thinking geliefert wurde | nur beschreibend |
| R-HIDDEN | ein abgeschlossener Stream mehr `completion_tokens` als sichtbare Zeichen meldet | nur beschreibend |
| R-LOOP | `finish_reason = length` oder ≥ 3 direkte Wiederholungen eines 20–200 Zeichen langen Stücks mit Buchstaben | Fehler |
| K-CORR | der nächste Request den Korrektur-Prompt des Harness enthält | bricht strict und funktional |
| I-ERR | ein Request mit Fehler endete (HTTP, Timeout, Abbruch) oder ein Stream ohne `finish_reason` und ohne abgefangenen Tag endete (Fehler-Chunk des Gateways, abgerissener Stream) | Lauf nicht in den Raten, getrennt gezählt |

**Erfolgsstufen.** Vollständig ist ein Lauf, wenn er das Ziel seiner Phase erreicht (−1: erster Call ausgeführt,
−2: beide Calls, −3: zusätzlich Endantwort). **strict:** vollständig, kein Fehler, kein A-LEAK, kein K-CORR.
**funktional:** wie strict, A-LEAK erlaubt (bis 17.11. Hauptmaßstab). **lenient** (Hauptmaß seit 17.11.,
Entscheidung Luca: die Korrektur gehört zum System): wie funktional, K-CORR erlaubt; die Codes
des Versuchs, der die Korrektur ausgelöst hat, zählen nicht, außer B-PREM und B-FORGE. **Sicht „bis Stage k“:**
dieselben Stufen nur über die Schritte ≤ k (vergleichbar mit Phase −k).

**Auslegung**
- Echo: gleiche Operation, nur das Attribut `id`, Id in dieser Eingabe veröffentlicht, Inhalt = Serverwert;
  Groß-/Kleinschreibung und Anführungszeichen egal. Jedes Zusatzattribut macht daraus B-FORGE.
- „Tag versucht“: jeder ADD/MUL-Tag mit Attribut, selbstschließend oder mit Schluss-Tag. Gültige Verweise
  (veröffentlichte Id), Echos und bloße Erwähnungen (`<ADD>`) zählen nicht; ein gefälschter Tag und ein Verweis
  auf eine nicht veröffentlichte Id zählen (B-FORGE bzw. A-WRONG statt A-OMIT; O11, bestätigt 15.11.). „Erkannt“ heißt:
  `harness.first_empty_tag` würde ihn ausführen.
- Der verworfene Rest des Chunks nach dem Tag zählt nicht. Wiederholt das Thinking das gesendete Resume-Präfix,
  wird es abgetrennt und als `prefix_echo` ausgewiesen (Indiz zu V0 c).
- Endantwort: Zahl nach dem letzten „=“, sonst die letzte Zahl im Antwortkanal (Tags entfernt, Echo-Inhalt bleibt).
- Raten und McNemar-Paarung: je Experiment und Aufgabe zählt der erste Lauf ohne I-ERR; weitere Läufe
  und ungepaarte Aufgaben werden ausgewiesen, nie mitgezählt (Entscheidung 15.11.). Ohne Paare gibt es keinen p-Wert.
- Ein Stream, der ohne `finish_reason` und ohne abgefangenen Tag endet, ist kein Modellverhalten: I-ERR statt
  A-OMIT (Entscheidung 15.11.). Die Stage bekommt dann keine weiteren Codes.
- Fragen außerhalb von `config/poc_items.json` sind nicht bewertbar (keine Codes, Hinweis).

### 6.2 Hilfsfunktionen: Zahlen, Tags, Eingaben

Tolerante Tag-Erkennung (Groß-/Kleinschreibung, Anführungszeichen, Bindestrich-Attribute) und Einordnung jedes Tags als call, placeholder, malformed, reference, dangling (Verweis auf nicht veröffentlichte Id), echo, forged
oder mention.

In [ ]:
FAILURE_CODES = ("A-OMIT", "A-MALF", "A-WRONG", "A-META", "B-PREM", "B-FORGE", "C-IGNORE", "C-FINAL", "R-LOOP")
DESCRIPTIVE_CODES = ("B-SELF", "R-EMPTY", "R-HIDDEN")      # reported, never decide a level
ALWAYS_COUNT = ("B-PREM", "B-FORGE")                      # count under lenient even in a corrected attempt
CODE_ORDER = ("A-OMIT", "A-MALF", "A-WRONG", "A-LEAK", "A-META", "B-PREM", "B-FORGE", "B-SELF",
              "C-IGNORE", "C-FINAL", "R-EMPTY", "R-HIDDEN", "R-LOOP", "K-CORR", "I-ERR")
ATTEMPT_KINDS = ("call", "placeholder", "malformed", "dangling", "forged")   # rule 5: a tag was attempted
CALL_KINDS = ("call", "placeholder", "malformed")                # call-like: leaks / extra calls


# --- numbers: whole numbers only, never a substring (plan §8; 18.5 and 0.18 are not 18) ---
NUMBER_TOKEN_RE = re.compile(r"(?<![\d.])\d+(?:\.\d+)?")


def number_re(value):
    return re.compile(r"(?<!\d)(?<!\d[.,])" + re.escape(harness._fmt(float(value))) + r"(?![.,]?\d)")


def contains_number(text, value):
    return number_re(value).search(text or "") is not None


def to_number(text):
    try:
        return float(str(text).strip())
    except ValueError:
        return None


def numbers_in(text):
    return [float(token) for token in NUMBER_TOKEN_RE.findall(text or "")]


# --- tolerant tag scanner: case, quote style and hyphenated attribute names do not hide a tag ---
TAG_START_RE = re.compile(r"<\s*(/)?\s*(add|mul)\b", re.IGNORECASE)
ATTR_TOKEN = r"""[\w-]+\s*=\s*(?:"[^"]*"|'[^']*'|[^\s"'<>/=]+)"""
OPEN_TAG_RE = re.compile(r"<\s*(add|mul)\b((?:\s+" + ATTR_TOKEN + r")*)\s*(/?)\s*>", re.IGNORECASE)
ATTR_ANY_RE = re.compile(r"""([\w-]+)\s*=\s*(?:"([^"]*)"|'([^']*)'|([^\s"'<>/=]+))""")
X_ARG_RE = re.compile(r"x\d+", re.IGNORECASE)
DEP_RE = re.compile(r"\s*<\s*DEP\s*>\s*")
ID_ATTR_RE = re.compile(r"""(\bid\s*=\s*)(?:"[^"]*"|'[^']*'|[^\s"'<>/=]+)""", re.IGNORECASE)


def scan_tags(text):
    """Every ADD/MUL tag: form 'self' (/>), 'pair' (with closing tag), 'open' or 'broken' (unparseable)."""
    text, tags, pos = text or "", [], 0
    while (start := TAG_START_RE.search(text, pos)) is not None:
        if start.group(1):  # a stray closing tag
            end = text.find(">", start.end())
            pos = start.end() if end < 0 else end + 1
            continue
        match = OPEN_TAG_RE.match(text, start.start())
        if match is None:
            end = text.find(">", start.end())
            end = min(len(text) if end < 0 else end + 1, start.start() + 200)
            tags.append({"name": start.group(2).upper(), "attrs": {}, "form": "broken", "body": None,
                         "raw": text[start.start():end], "start": start.start(), "end": end})
            pos = end
            continue
        name = match.group(1)
        attrs = {m.group(1): next(v for v in m.groups()[1:] if v is not None)
                 for m in ATTR_ANY_RE.finditer(match.group(2))}
        form, body, end = ("self" if match.group(3) else "open"), None, match.end()
        if form == "open":
            close = re.compile(r"([^<]*)<\s*/\s*" + name + r"\s*>", re.IGNORECASE).match(text, end)
            if close:
                form, body, end = "pair", close.group(1), close.end()
        tags.append({"name": name.upper(), "attrs": attrs, "form": form, "body": body,
                     "raw": text[start.start():end], "start": start.start(), "end": end})
        pos = end
    return tags


def classify_tag(tag, published):
    """Kind of a tag (rules 4 and 5): call, placeholder, malformed, reference, dangling, echo, forged or mention.

    published: {id: (op, value)} visible in this turn's input. A call is what the harness itself would
    intercept (harness.first_empty_tag); everything call-like that it would not intercept is malformed.
    """
    attrs = {key.lower(): value for key, value in tag["attrs"].items()}
    known = published.get(attrs.get("id"))
    id_only = set(attrs) == {"id"}
    if tag["form"] == "pair" and tag["body"].strip():
        same = known is not None and known[0] == tag["name"] and to_number(tag["body"]) == known[1]
        return "echo" if id_only and same else "forged"
    if id_only:
        # a reference names a result published for the same operation; otherwise it dangles (A-WRONG, O11)
        return "reference" if known is not None and known[0] == tag["name"] else "dangling"
    if tag["form"] == "self":
        recognized = harness.first_empty_tag(tag["raw"])
        if recognized is not None and recognized["raw"] == tag["raw"]:
            x_values = [v for k, v in recognized["attrs"].items() if re.fullmatch(r"x\d+", k)]
            placeholder = any(to_number(v) is None and not DEP_RE.fullmatch(v) for v in x_values)
            return "placeholder" if placeholder else "call"
        return "malformed"
    if tag["form"] == "open" and not attrs:
        return "mention"
    if tag["form"] == "broken" and "=" not in tag["raw"]:
        return "mention"
    return "malformed"


def mask_ids(text):
    """Blank id attribute values (mul-1, add-2) so their digits are not read as numbers."""
    return ID_ATTR_RE.sub(r'\1""', text or "")


def strip_tags(text):
    """Text with tags removed; a closed pair keeps its body (used for the final value only)."""
    text, out, pos = text or "", [], 0
    for tag in scan_tags(text):
        out.append(text[pos:tag["start"]])
        out.append(f" {tag['body']} " if tag["form"] == "pair" else " ")
        pos = tag["end"]
    out.append(text[pos:])
    return "".join(out)


def final_value(answer):
    """Rule 9: the number after the last '=', else the last number in the answer channel."""
    text = strip_tags(answer)
    if "=" in text:
        after = numbers_in(text[text.rindex("=") + 1:])
        if after:
            return after[0]
    numbers = numbers_in(text)
    return numbers[-1] if numbers else None


LOOP_RE = re.compile(r"(.{20,200}?)\1{2,}", re.DOTALL)


def has_loop(text):
    """Rule 10: a 20-200 character unit containing letters, repeated at least 3 times in a row."""
    return any(re.search(r"[^\W\d_]", m.group(1)) for m in LOOP_RE.finditer(text or ""))


# --- what a turn received ---
RESULT_ENTRY_RE = re.compile(r'<(ADD|MUL) id="([^"]+)">([^<]*)</\1>')
PREFIX_MARKER_RE = re.compile(r"^(?:\s*(?:<\|im_start\|>(?:assistant\b)?|<think>))*\s*")


def published_results(messages):
    """{id: (op, value)} from the RESULTS message of a turn's input, in publication order."""
    for message in messages:
        if message["role"] == "user" and message["content"].startswith("<RESULTS>"):
            return {m.group(2): (m.group(1), to_number(m.group(3)))
                    for m in RESULT_ENTRY_RE.finditer(message["content"])}
    return {}


def has_correction(messages):
    """The harness's correction prompt: a user message after the question that is not RESULTS."""
    return any(index >= 2 and m["role"] == "user" and not m["content"].startswith("<RESULTS>")
               for index, m in enumerate(messages))


def sent_prefix(messages):
    """Resume prefix of a turn (assistant partial message) without <|im_start|>/<think>; None if none."""
    if messages and messages[-1]["role"] == "assistant":
        return PREFIX_MARKER_RE.sub("", messages[-1]["content"], count=1)
    return None


def find_item(record):
    """Item of a run (rule 12): by expected['item_id'] if present, else by the exact question text.

    Test-suite runs (experiment "suite:<file>", section 5.4): the item is computed from the question itself.
    """
    expected = record.get("expected")
    if str(record.get("experiment") or "").startswith("suite:"):
        if "suite_item" not in globals():
            return None, "suite run: run cell 5.4 first"
        try:
            item = suite_item(record.get("question") or "", (expected or {}).get("item_id"), record["experiment"])
        except ValueError as error:
            return None, f"suite question not evaluable: {error}"
        if isinstance(expected, dict) and expected.get("answer") not in (None, item["ground_truth"]):
            return None, f"suite: stored answer {expected.get('answer')} != computed {item['ground_truth']}"
        return item, None
    item_id = expected.get("item_id") if isinstance(expected, dict) else None
    if item_id is not None:
        item = ITEMS.get(item_id)
        if item is None:
            return None, f"unknown item id {item_id}"
        if item["question"] != record.get("question"):
            return None, f"question differs from item {item_id}"
        return item, None
    item = ITEMS_BY_QUESTION.get((record.get("question") or "").strip())
    return (item, None) if item else (None, "question not in config/poc_items.json")


def expected_call(item, step, published_values):
    """(op, sorted args) of call `step`; the second call must use the value the server actually published."""
    call = item["expected_calls"][step - 1]
    args = [float(a) for a in call["args"]]
    if step == 2 and published_values:
        args[args.index(float(item["expected_calls"][0]["result"]))] = published_values[0]
    return call["op"], sorted(args)


def call_args(attrs, dep_value):
    """x-arguments resolved like the harness does (<DEP> = last server value); None marks unresolvable."""
    dep = dep_value if dep_value is not None else harness.NO_RESULT
    keys = sorted((k for k in attrs if re.fullmatch(r"x\d+", k)), key=lambda k: int(k[1:]))
    return [harness.parse_number(attrs[k], dep) for k in keys]


def fmt_call(op, args):
    shown = ["?" if a is None else harness._fmt(a) for a in args]
    return f"{op}({', '.join(shown)})"

### 6.3 Bewertung eines Laufs: `evaluate_run(record)`

Codes je Stage und Lauf, Endantwort, Erfolgsstufen und Sicht „bis Stage k“. Aus `interrupt` wird nur übernommen, welcher Tag ausgeführt wurde; die eigene Tag-Erkennung wird damit abgeglichen (Abweichung → Hinweis).

In [ ]:
def analysed_texts(result):
    """Rule 6: output up to the stop. The chunk rest after the intercepted tag was discarded and is not read."""
    thinking, answer = result["reasoning"] or "", result["content_all"] or ""
    interrupt = result["interrupt"]
    if interrupt and interrupt.get("leak"):
        answer = answer[:interrupt["tool"]["end"]]
    return thinking, answer


def _add(stage, code, detail=""):
    stage["codes"].append({"code": code, "detail": detail})


STREAM_INCOMPLETE = "stream ended without finish_reason and without an intercepted tag"


def _fmt_values(values):
    return ", ".join(harness._fmt(v) for v in values)


def _evaluate_stage(turn, item, question_numbers, notes):
    result, messages = turn.get("result"), turn["messages"]
    published = published_results(messages)
    pub_values = [value for _, value in published.values()]
    final = not turn["params"]["stop_at_tool"]
    stage = {
        "stage": turn["stage"], "step": len(item["expected_calls"]) + 1 if final else len(published) + 1,
        "final": final, "after_correction": has_correction(messages), "published": dict(published),
        "codes": [], "tags": [], "call": None, "prefix_echo": None, "answer": None, "error": turn.get("error"),
    }
    if result is None:
        _add(stage, "I-ERR", json.dumps(turn.get("error"), ensure_ascii=False))
        return stage
    if result["interrupt"] is None and result.get("finish_reason") is None:
        # a completed stream always reports a finish_reason: this one was cut or carried only an error chunk
        _add(stage, "I-ERR", STREAM_INCOMPLETE)
        return stage
    thinking, answer = analysed_texts(result)
    prefix = sent_prefix(messages)
    if prefix is not None and prefix.strip():
        stage["prefix_echo"] = thinking.lstrip().startswith(prefix.strip())
        if stage["prefix_echo"]:  # rule 6: the repeated prefix was evaluated in the previous stage
            thinking = thinking.lstrip()[len(prefix.strip()):]
    stage["answer"] = answer
    tags = [dict(tag, channel=channel, kind=classify_tag(tag, published))
            for channel, text in (("thinking", thinking), ("answer", answer)) for tag in scan_tags(text)]
    stage["tags"] = tags
    interrupt = result["interrupt"]

    # cross-check: the first call the scanner recognizes must be the tag the harness executed
    if not final:
        calls = [t for t in tags if t["kind"] in ("call", "placeholder")]
        first = next((t for t in calls if t["channel"] == "thinking"), None) or (calls[0] if calls else None)
        mine, theirs = (first["raw"] if first else None), (interrupt["tool"]["raw"] if interrupt else None)
        if mine != theirs:
            notes.append(f"Stage {turn['stage']}: Scanner {mine!r} ≠ Harness {theirs!r}")

    def raws(kinds, channel=None):
        return [t["raw"] for t in tags if t["kind"] in kinds and channel in (None, t["channel"])]

    if not final and not raws(ATTEMPT_KINDS):
        _add(stage, "A-OMIT", "no tool tag attempted")
    if raws(("malformed",)):
        _add(stage, "A-MALF", " · ".join(raws(("malformed",))))
    if raws(("dangling",)):
        _add(stage, "A-WRONG", "reference to an unpublished id: " + " · ".join(raws(("dangling",))))
    if final and raws(CALL_KINDS):
        _add(stage, "A-WRONG", "extra call after both results: " + " · ".join(raws(CALL_KINDS)))
    if raws(CALL_KINDS, "answer"):
        _add(stage, "A-LEAK", " · ".join(raws(CALL_KINDS, "answer")))
    if interrupt is not None:
        tool = interrupt["tool"]
        args = call_args(tool["attrs"], turn["params"]["dep_value"])
        stage["call"] = {"raw": tool["raw"], "op": tool["name"], "args": args,
                         "value": interrupt["value"], "leak": bool(interrupt["leak"])}
        x_values = [v for k, v in tool["attrs"].items() if re.fullmatch(r"x\d+", k)]
        if any(to_number(v) is None and not DEP_RE.fullmatch(v) for v in x_values):
            _add(stage, "A-META", f"placeholder intercepted: {tool['raw']}")
        elif None in args:
            _add(stage, "A-WRONG", f"order: <DEP> without a server value in {tool['raw']}")
        else:
            got = (tool["name"], sorted(args))
            want = expected_call(item, stage["step"], pub_values)
            if got != want:
                detail = f"expected {fmt_call(*want)}, got {fmt_call(*got)}"
                other = [s for s in range(1, len(item["expected_calls"]) + 1)
                         if s != stage["step"] and expected_call(item, s, pub_values) == got]
                _add(stage, "A-WRONG", detail + (f" (call of step {other[0]}: order)" if other else ""))
            if pub_values and pub_values[-1] not in args:
                _add(stage, "C-IGNORE", f"call does not use the last server value {harness._fmt(pub_values[-1])}")
    results = [float(call["result"]) for call in item["expected_calls"]]
    pending = [r for r in dict.fromkeys(results) if r not in pub_values and r not in question_numbers]
    premature = [r for r in pending if contains_number(mask_ids(answer), r)]
    if premature:
        _add(stage, "B-PREM", f"{_fmt_values(premature)} in the answer channel before the server published it")
    if raws(("forged",)):
        _add(stage, "B-FORGE", " · ".join(f"{t['raw']} ({t['channel']})" for t in tags if t["kind"] == "forged"))
    self_computed = [r for r in pending if contains_number(mask_ids(thinking), r)]
    if self_computed:
        _add(stage, "B-SELF", f"{_fmt_values(self_computed)} in the thinking before the server published it")
    if not (result["reasoning_all"] or "").strip():
        _add(stage, "R-EMPTY", "no thinking delivered")
    tokens = (result.get("usage") or {}).get("completion_tokens")
    visible = len(result["reasoning_all"] or "") + len(result["content_all"] or "")
    if interrupt is None and result.get("finish_reason") in ("stop", "length") and tokens is not None and tokens > visible:
        _add(stage, "R-HIDDEN", f"{tokens} completion_tokens > {visible} visible characters")
    if result.get("finish_reason") == "length":
        _add(stage, "R-LOOP", "finish_reason = length")
    elif has_loop(thinking) or has_loop(answer):
        _add(stage, "R-LOOP", "repetition loop")
    return stage


def _judge(stages, complete, upto):
    """Levels over the stages of steps <= upto (plan §8; rules 1-3)."""
    relevant = [stage for stage in stages if stage["step"] <= upto]
    codes = [(c["code"], stage) for stage in relevant for c in stage["codes"]]
    corrected_attempt = lambda stage: any(c["code"] == "K-CORR" for c in stage["codes"])
    failure = any(code in FAILURE_CODES for code, _ in codes)
    leak = any(code == "A-LEAK" for code, _ in codes)
    correction = any(code == "K-CORR" for code, _ in codes)
    lenient_failure = any(code in FAILURE_CODES and (code in ALWAYS_COUNT or not corrected_attempt(stage))
                          for code, stage in codes)
    return {
        "strict": complete and not failure and not leak and not correction,
        "functional": complete and not failure and not correction,
        "lenient": complete and not lenient_failure,
    }


def level_label(levels):
    if not levels:
        return None
    for key, label in (("strict", "strict"), ("functional", "funktional"), ("lenient", "lenient")):
        if levels[key]:
            return label
    return "✗"


def evaluate_run(record):
    """Codes per stage and per run, computed from the raw data only (the harness verdict is not used)."""
    item, problem = find_item(record)
    errors = [turn["error"] for turn in record["turns"] if turn.get("error")]
    infra = record.get("error") or (errors[0] if errors else None)
    if infra is None and not record["turns"]:
        infra = {"type": "NoTurns", "message": "no request was recorded"}
    ev = {
        "run_id": record["run_id"], "started_at": record["started_at"], "experiment": record["experiment"],
        "phase": record["phase"], "model": record["model"], "prompt_version": record["prompt_version"],
        "question": record["question"], "item_id": item["id"] if item else None,
        "item_set": item["set"] if item else None, "ground_truth": item["ground_truth"] if item else None,
        "status": "I-ERR" if infra else ("ok" if item else "not evaluable"),
        "infra_error": infra, "notes": [] if item else [problem], "stages": [], "calls": [],
        "final_text": None, "final_value": None, "last_server_value": None, "correct": None,
        "matches_server": None, "corrected": False, "complete": False, "levels": None, "through": {},
    }
    if item is None:
        return ev
    question_numbers = set(numbers_in(record["question"]))
    stages = [_evaluate_stage(turn, item, question_numbers, ev["notes"]) for turn in record["turns"]]
    if infra is None and any(c["code"] == "I-ERR" for stage in stages for c in stage["codes"]):
        ev.update(status="I-ERR", infra_error={"type": "StreamIncomplete", "message": STREAM_INCOMPLETE})
    for stage, following in zip(stages, stages[1:]):
        if following["after_correction"]:
            _add(stage, "K-CORR", "the next request carried the harness's correction prompt")
    executed = [stage["call"] for stage in stages if stage["call"] and stage["call"]["value"] is not None]
    ev.update(stages=stages, calls=executed, corrected=any(s["after_correction"] for s in stages))
    ev["last_server_value"] = executed[-1]["value"] if executed else None
    n_calls = len(item["expected_calls"])
    last = stages[-1] if stages else None
    if record["phase"] == 3 and last and last["answer"] is not None and last["call"] is None \
            and (last["final"] or numbers_in(strip_tags(last["answer"]))):
        value, server = final_value(last["answer"]), ev["last_server_value"]
        ev.update(final_text=last["answer"].strip(), final_value=value,
                  correct=value is not None and value == item["ground_truth"],
                  matches_server=value is not None and server is not None and value == server)
        problems = []
        if value is None:
            problems.append("no number in the final answer")
        else:
            if value != item["ground_truth"]:
                problems.append(f"{harness._fmt(value)} ≠ correct result {item['ground_truth']}")
            if server is None or value != server:
                problems.append(f"{harness._fmt(value)} ≠ last server value "
                                f"{'–' if server is None else harness._fmt(server)}")
        if problems:
            _add(last, "C-FINAL", "; ".join(problems))
    if record["phase"] < 3 and ev["status"] == "ok":  # an infrastructure error leaves correctness open
        ev["correct"] = [(c["op"], sorted(c["args"])) for c in executed[:record["phase"]]] == \
            [(c["op"], sorted(map(float, c["args"]))) for c in item["expected_calls"][:record["phase"]]]
    goal = min(record["phase"], n_calls)
    ev["complete"] = len(executed) >= goal and (record["phase"] < 3 or bool(last and last["final"]))
    if ev["status"] == "ok":
        ev["levels"] = _judge(stages, ev["complete"], record["phase"])
        for upto in range(1, record["phase"] + 1):
            done = len(executed) >= upto if upto <= n_calls else ev["complete"]
            ev["through"][upto] = _judge(stages, done, upto)
    return ev

### 6.4 Urteil des Harness (nur zum Vergleich)

Aus `harness_log`: Guard je Stage (prüft nur das Thinking und erst ab Stage 2), „quoted in answer“ (Teilstring-Vergleich) und `[halt]`.

In [ ]:
STAGE_LABEL_RE = re.compile(r"^-{78}\nSTAGE (\d+) — (.+)\n-{78}$", re.MULTILINE)
GUARD_RE = re.compile(r"^\[guard\] forged tool results in model output : (.*)$", re.MULTILINE)
QUOTED_RE = re.compile(r"quoted in answer: (True|False)$", re.MULTILINE)
HALT_TEXT = "[halt] intercepted call has an unresolved <DEP>"


def harness_verdict(record):
    """The harness's own verdict, read from its log: guard per stage, 'quoted in answer', halt.

    Shown next to the own codes only; it never enters them. Markers are taken from the end of each
    stage block, after the model text the harness printed.
    """
    log = record.get("harness_log") or ""
    marks = list(STAGE_LABEL_RE.finditer(log))
    stages = {}
    for index, mark in enumerate(marks):
        block = log[mark.end(): marks[index + 1].start() if index + 1 < len(marks) else len(log)]
        guards = GUARD_RE.findall(block)
        stages[int(mark.group(1))] = {
            "label": mark.group(2).strip(),
            "guard": guards[-1].strip() if guards else None,   # None: stage 1, the guard does not run
            "halt": HALT_TEXT in block,
        }
    quoted = QUOTED_RE.findall(log)
    return {
        "stages": stages,
        "guard_flagged": any(s["guard"] not in (None, "none") for s in stages.values()),
        "quoted": (quoted[-1] == "True") if quoted else None,
        "halt": any(s["halt"] for s in stages.values()),
    }

### 6.5 Darstellung

`show_evaluation(record)`: Bewertung je Stage mit Harness-Urteil daneben · `evaluation_frame()`: eine Zeile pro Lauf · Lauftabelle, Abdeckung, Codes je Schritt.

In [ ]:
from collections import Counter
from zoneinfo import ZoneInfo

BERLIN = ZoneInfo("Europe/Berlin")
LEVEL_KEYS = ("strict", "functional", "lenient")
LEVEL_NAMES = {"strict": "strict", "functional": "funktional", "lenient": "lenient"}
STATUS_NAMES = {"ok": "bewertet", "I-ERR": "I-ERR (Infrastruktur, nicht in den Raten)",
                "not evaluable": "nicht bewertbar"}


def berlin_time(iso):
    return datetime.fromisoformat(iso).astimezone(BERLIN).strftime("%d.%m.%Y %H:%M")


def mark(value):
    return "–" if value is None else ("✓" if value else "✗")


def find_run(run_id, paths=None):
    """A stored run by id, from runs.jsonl, playground.jsonl or selftest.jsonl."""
    for path in paths or (RUNS_PATH, PLAYGROUND_PATH, SELFTEST_PATH):
        for record in read_jsonl(path):
            if record["run_id"] == run_id:
                return record
    raise KeyError(f"run {run_id} not found")


def evaluate_file(path, source):
    """Evaluate every record of a JSONL file (unfiltered): record, own evaluation, harness verdict."""
    return [{"source": source, "record": record, "ev": evaluate_run(record), "harness": harness_verdict(record)}
            for record in read_jsonl(path)]


def codes_by_stage(ev, descriptive=False):
    parts = []
    for stage in ev["stages"]:
        codes = {c["code"] for c in stage["codes"] if (c["code"] in DESCRIPTIVE_CODES) == descriptive}
        if codes:
            parts.append(f"S{stage['stage']}: " + ", ".join(sorted(codes, key=CODE_ORDER.index)))
    text = "; ".join(parts)
    if not descriptive and ev["status"] == "I-ERR" and "I-ERR" not in text:
        text = "I-ERR" + ("; " + text if text else "")
    return text


def _expected_and_answer(ev):
    if ev["item_id"] is None:
        return "–", "–"
    if ev["phase"] == 3:
        answer = "–" if ev["final_value"] is None else harness._fmt(ev["final_value"])
        return str(ev["ground_truth"]), answer
    item = ITEMS[ev["item_id"]]
    expected = " → ".join(fmt_call(c["op"], c["args"]) for c in item["expected_calls"][:ev["phase"]])
    answer = " → ".join(fmt_call(c["op"], c["args"]) for c in ev["calls"]) or "–"
    return expected, answer


def _prefix_echo_text(ev):
    sent = [s for s in ev["stages"] if s["prefix_echo"] is not None]
    if not sent:
        return "–"
    echoed = [f"S{s['stage']}" for s in sent if s["prefix_echo"]]
    return "ja (" + ", ".join(echoed) + ")" if echoed else "nein"


def evaluation_frame(evaluated):
    """One row per run: own evaluation, levels, through-step view and the harness verdict next to it."""
    rows = []
    for entry in evaluated:
        record, ev, verdict = entry["record"], entry["ev"], entry["harness"]
        expected, answer = _expected_and_answer(ev)
        row = {
            "source": entry["source"], "run_id": ev["run_id"], "started_at": ev["started_at"],
            "time": berlin_time(ev["started_at"]), "experiment": ev["experiment"], "phase": ev["phase"],
            "model": ev["model"], "prompt_version": ev["prompt_version"], "item_id": ev["item_id"],
            "item_set": ev["item_set"] or "–", "question": ev["question"],
            "task": f"{ev['item_id'] or '–'} · {ev['question'].replace('Please calculate ', '')}",
            "status": ev["status"], "expected": expected, "answer": answer, "correct": ev["correct"],
            "correct_mark": mark(ev["correct"]) if ev["status"] != "not evaluable" else "–",
            "level": level_label(ev["levels"]) or ev["status"],
            "failure_codes": codes_by_stage(ev), "descriptive_codes": codes_by_stage(ev, descriptive=True),
            "prefix_echo": _prefix_echo_text(ev), "corrected": ev["corrected"],
            "harness_guard": "Fälschung gemeldet" if verdict["guard_flagged"] else "keine",
            "harness_quoted": verdict["quoted"], "notes": "; ".join(ev["notes"]),
        }
        for key in LEVEL_KEYS:
            row[key] = ev["levels"][key] if ev["levels"] else None
            for upto in (1, 2, 3):
                row[f"thru{upto}_{key}"] = ev["through"][upto][key] if upto in ev["through"] else None
        rows.append(row)
    return pd.DataFrame(rows)


RUN_TABLE_COLUMNS = {
    "run_id": "Lauf", "time": "Zeit", "experiment": "Experiment", "phase": "Phase", "model": "Modell",
    "prompt_version": "Prompt", "task": "Aufgabe", "expected": "erwartet", "answer": "Antwort",
    "correct_mark": "korrekt", "level": "Stufe", "failure_codes": "Codes", "descriptive_codes": "beschreibend",
    "prefix_echo": "prefix_echo",
}


def show_run_table(frame):
    """Presentation table, one row per run; raw output and details via show_run / show_evaluation."""
    table = frame[list(RUN_TABLE_COLUMNS)].rename(columns=RUN_TABLE_COLUMNS)
    table["Phase"] = table["Phase"].map(lambda p: f"−{p}")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(table.set_index("Lauf"))
    display(HTML(
        "<p><i>Rohausgaben eines Laufs (gesendete Nachrichten, Thinking, Antwort): "
        "<code>show_run(find_run(\"&lt;Lauf&gt;\"))</code> · Bewertung je Stage mit Harness-Urteil daneben: "
        "<code>show_evaluation(find_run(\"&lt;Lauf&gt;\"))</code>. Codes: <code>S2: A-LEAK</code> = Code in Stage 2; "
        "Stufen und Codes siehe 6.1.</i></p>"
    ))


def _grid(header, rows):
    head = "".join(f'<th style="text-align:left">{html.escape(str(h))}</th>' for h in header)
    body = "".join(
        "<tr>" + "".join(f'<td style="text-align:left;vertical-align:top">{cell}</td>' for cell in row) + "</tr>"
        for row in rows
    )
    return f"<table><tr>{head}</tr>{body}</table>"


def render_evaluation(record):
    ev, verdict = evaluate_run(record), harness_verdict(record)
    expected, answer = _expected_and_answer(ev)
    levels = ev["levels"]
    level_text = (f"{level_label(levels)} (strict {mark(levels['strict'])} · funktional {mark(levels['functional'])}"
                  f" · lenient {mark(levels['lenient'])})") if levels else STATUS_NAMES[ev["status"]]
    through = " · ".join(f"bis Stage {k}: {mark(v['functional'])}" for k, v in ev["through"].items()) or "–"
    head = [
        ("Aufgabe", f"{ev['item_id'] or '–'} · {ev['question']}"),
        ("Phase · Modell · Prompt", f"−{ev['phase']} · {ev['model']} · {ev['prompt_version']}"),
        ("Status", STATUS_NAMES[ev["status"]] + (f": {ev['infra_error']}" if ev["infra_error"] else "")),
        ("Stufe", level_text),
        ("Sicht bis Stage (funktional)", through),
        ("erwartet · Antwort", f"{expected} · {answer}"),
        ("= richtiges Ergebnis", mark(ev["correct"])),
        ("= letzter Serverwert", mark(ev["matches_server"]) if ev["phase"] == 3 else "–"),
        ("Harness: quoted in answer", "–" if verdict["quoted"] is None else verdict["quoted"]),
        ("Korrektur-Prompt", "ja" if ev["corrected"] else "nein"),
        ("Hinweise", "; ".join(ev["notes"]) or "–"),
    ]
    rows = []
    for stage in ev["stages"]:
        call = stage["call"]
        intercepted = "–" if call is None else html.escape(
            f"{call['raw']} · {'Antwortkanal (Leak)' if call['leak'] else 'Thinking'} · "
            f"{'nicht auflösbar' if call['value'] is None else harness._fmt(call['value'])}")
        tags = "<br>".join(html.escape(f"{t['kind']}: {t['raw']} ({t['channel']})")
                           for t in stage["tags"] if t["kind"] != "mention") or "–"
        codes = "<br>".join(f"<b>{c['code']}</b> {html.escape(c['detail'])}" for c in stage["codes"]) or "–"
        guard = verdict["stages"].get(stage["stage"], {}).get("guard")
        rows.append([
            stage["stage"], "Endantwort" if stage["final"] else f"Call {stage['step']}",
            "nach Korrektur" if stage["after_correction"] else "", intercepted, tags, codes,
            "– (Stage 1: kein Guard)" if guard is None else html.escape(guard),
            {None: "–", True: "ja", False: "nein"}[stage["prefix_echo"]],
        ])
    parts = [
        f"<h3>Bewertung Lauf {html.escape(ev['run_id'])}</h3>", _table(head),
        _grid(["Stage", "Schritt", "Eingabe", "abgefangener Tag · Kanal · Wert", "Tags (eigene Erkennung)",
               "eigene Codes", "Harness-Guard", "prefix_echo"], rows),
        f"<p><i>Rohausgaben: <code>show_run(find_run(\"{html.escape(ev['run_id'])}\"))</code></i></p>",
    ]
    return "\n".join(parts)


def show_evaluation(record):
    display(HTML(render_evaluation(record)))


def code_counts(evaluated):
    """Number of evaluated runs with a code, per experiment and step (1, 2 = calls, 3 = final answer)."""
    counts, runs = Counter(), Counter()
    for entry in evaluated:
        ev = entry["ev"]
        if ev["status"] != "ok":
            continue
        runs[ev["experiment"]] += 1
        counts.update({(ev["experiment"], c["code"], stage["step"]) for stage in ev["stages"] for c in stage["codes"]})
    rows = [
        {"Experiment": f"{experiment} (n = {runs[experiment]})", "Code": code,
         **{f"Schritt {step}": counts.get((experiment, code, step), 0) for step in (1, 2, 3)}}
        for experiment in sorted(runs) for code in CODE_ORDER
        if any(counts.get((experiment, code, step)) for step in (1, 2, 3))
    ]
    return pd.DataFrame(rows).set_index(["Experiment", "Code"]) if rows else None


def coverage_table(frame):
    """Task × prompt × model: number of runs and the last time (Europe/Berlin); I-ERR runs marked."""
    if frame.empty:
        return None
    combos = sorted({f"{p} · {m}" for p, m in zip(frame["prompt_version"], frame["model"])})
    tasks = [f"{item['id']} · {item['question'].replace('Please calculate ', '')}" for item in ITEMS.values()]
    tasks += sorted(set(frame["task"]) - set(tasks))
    cells = {}
    for (task, prompt, model), group in frame.groupby(["task", "prompt_version", "model"]):
        ierr = int((group["status"] == "I-ERR").sum())
        last = berlin_time(group["started_at"].max())
        cells[(task, f"{prompt} · {model}")] = f"{len(group)}× · {last}" + (f" ({ierr} I-ERR)" if ierr else "")
    table = pd.DataFrame([[cells.get((task, combo), "–") for combo in combos] for task in tasks],
                         index=pd.Index(tasks, name="Aufgabe"), columns=combos)
    return table

### 6.6 Statistik

Raten je Zelle (Experiment × Prompt × Modell × Aufgabenset × Phase) mit 95-%-Wilson-Intervall, je Experiment und Aufgabe nur über den ersten gültigen Lauf (weitere Läufe werden ausgewiesen); gepaarte Vergleiche mit exaktem McNemar-Test (`math.comb`, ohne scipy); Balkendiagramm mit matplotlib (Beschriftung:
k/n der Stufe lenient, Hauptmaß seit 17.11.).

In [ ]:
import matplotlib.pyplot as plt

WILSON_Z = 1.959963984540054  # two-sided 95 %
CELL_KEYS = ("experiment", "prompt_version", "model", "item_set", "phase")
# plan §7: baseline first; E5 uses other tasks (T2), so it has no pairs with E2
PLANNED_COMPARISONS = [("E1", "E2", "Prompt S0 → S1"), ("E1b", "E4", "Prompt S0 → S1 auf M2 (Ersatz)"),
                       ("E4", "E7", "Prompt S1 → S3 auf M2"), ("E1b", "E7", "Prompt S0 → S3 auf M2"),
                       ("E7", "E7.1", "Prompt S3 → S3.1 auf M2"),
                       ("E1", "E9", "Prompt S0 → S3.1 auf M1"), ("E7.1", "E9", "Modell M2 → M1 mit S3.1"),
                       ("E2", "E4", "Modell M1 → M2"),
                       ("E2", "E3", "Prompt S1 → S2"), ("E2", "E5", "Aufgaben T1 → T2"),
                       ("E2", "E6", "falsche Serverwerte")]
# ordinal blue ramp (levels are nested: strict ⊂ funktional ⊂ lenient); ink and surface for the chart
LEVEL_COLORS = {"strict": "#184f95", "functional": "#2a78d6", "lenient": "#86b6ef"}
INK = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#898781", "grid": "#e1e0d9",
       "axis": "#c3c2b7", "surface": "#fcfcfb"}


def wilson_interval(k, n, z=WILSON_Z):
    """Wilson score interval for k successes in n runs; (nan, nan) for n = 0."""
    if n == 0:
        return math.nan, math.nan
    p = k / n
    denominator = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denominator
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
    return max(0.0, center - half), min(1.0, center + half)


def mcnemar_exact(b, c):
    """Exact two-sided McNemar test (binomial on the b + c discordant pairs, p = 0.5); no scipy."""
    n = b + c
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def fmt_rate(k, n):
    if n == 0:
        return "–"
    low, high = wilson_interval(k, n)
    return f"{k}/{n} · {k / n:.2f} [{low:.2f}; {high:.2f}]".replace(".", ",")


def cell_label(row, multiline=False):
    parts = [f"{row['experiment']} · {row['prompt_version']}", row["model"],
             f"{row['item_set']} · −{row['phase']}"]
    return ("\n" if multiline else " · ").join(parts)


def first_valid_runs(frame):
    """Per experiment and task the first run without I-ERR (by start time); repeats are never rated."""
    valid = frame[frame["status"] == "ok"].sort_values("started_at", kind="stable")
    return valid.drop_duplicates(["experiment", "item_id"], keep="first")


def rate_table(frame, by=CELL_KEYS):
    """Per cell: runs, I-ERR, not evaluable, further runs, and k/n with Wilson CI per level.

    n counts the first valid run per experiment and task only; I-ERR is never rated.
    """
    rows = []
    for keys, group in frame.groupby(list(by), sort=True, dropna=False):
        rated = first_valid_runs(group)
        n = len(rated)
        row = dict(zip(by, keys), runs=len(group), ierr=int((group["status"] == "I-ERR").sum()),
                   not_evaluable=int((group["status"] == "not evaluable").sum()),
                   extra=int((group["status"] == "ok").sum()) - n, n=n)
        for key in LEVEL_KEYS:
            k = int(rated[key].astype(bool).sum())
            low, high = wilson_interval(k, n)
            row.update({f"{key}_k": k, f"{key}_rate": k / n if n else math.nan, f"{key}_lo": low, f"{key}_hi": high})
        for upto in (1, 2, 3):
            column = rated[f"thru{upto}_functional"].dropna()
            row[f"thru{upto}_k"], row[f"thru{upto}_n"] = int(column.astype(bool).sum()), len(column)
        rows.append(row)
    return pd.DataFrame(rows)


def show_rate_table(table):
    records = table.to_dict("records")
    shown = pd.DataFrame({
        "Zelle": [cell_label(r) for r in records],
        "Läufe": [r["runs"] for r in records],
        "I-ERR": [r["ierr"] for r in records],
        "nicht bewertbar": [r["not_evaluable"] for r in records],
        "weitere Läufe": [r["extra"] for r in records],
        **{LEVEL_NAMES[key]: [fmt_rate(r[f"{key}_k"], r["n"]) for r in records] for key in LEVEL_KEYS},
        **{f"bis Stage {u} (funkt.)": [fmt_rate(r[f"thru{u}_k"], r[f"thru{u}_n"]) for r in records] for u in (1, 2, 3)},
    }).set_index("Zelle")
    with pd.option_context("display.max_colwidth", None):
        display(shown)


def plot_rates(table, title):
    """Grouped bars per cell (strict / funktional / lenient) with Wilson 95 % CI whiskers."""
    rated = table[table["n"] > 0].reset_index(drop=True)
    if rated.empty:
        print(f"{title}: keine bewerteten Läufe – kein Diagramm")
        return None
    count, dpi = len(rated), 100
    width_in = max(4.5, 1.7 * count + 1.2)
    fig, ax = plt.subplots(figsize=(width_in, 4.0), dpi=dpi)
    fig.subplots_adjust(left=0.12, right=0.98, bottom=0.26, top=0.78)
    fig.patch.set_facecolor(INK["surface"])
    ax.set_facecolor(INK["surface"])
    ax.set_xlim(-0.6, count - 0.4)
    units_per_px = (count + 0.2) / (width_in * dpi * 0.86)
    bar, gap = min(0.26, 24 * units_per_px), 2 * units_per_px      # bars <= 24 px, 2 px apart
    for index, key in enumerate(LEVEL_KEYS):
        xs = [i + (index - 1) * (bar + gap) for i in range(count)]
        rates = rated[f"{key}_rate"]
        ax.bar(xs, rates, bar, color=LEVEL_COLORS[key], label=LEVEL_NAMES[key], zorder=2)
        # float rounding: at 0/n and n/n the Wilson bound can lie 1e-16 inside the rate; whiskers must be >= 0
        lower = (rates - rated[f"{key}_lo"]).clip(lower=0)
        upper = (rated[f"{key}_hi"] - rates).clip(lower=0)
        ax.errorbar(xs, rates, yerr=[lower, upper], fmt="none",
                    ecolor=INK["secondary"], elinewidth=1, capsize=2.5, zorder=3)
    for i, row in enumerate(rated.to_dict("records")):   # selective label: k/n of the main measure (lenient, §8)
        ax.annotate(f"{row['lenient_k']}/{row['n']}", (i + bar + gap, row["lenient_hi"]), xytext=(0, 3),
                    textcoords="offset points", ha="center", va="bottom", fontsize=8, color=INK["secondary"])
    ax.set_xticks(range(count))
    ax.set_xticklabels([cell_label(r, multiline=True) for r in rated.to_dict("records")], fontsize=8,
                       color=INK["secondary"])
    ax.set_ylim(0, 1.12)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0 %", "25 %", "50 %", "75 %", "100 %"])
    ax.yaxis.grid(True, color=INK["grid"], linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(INK["axis"])
    ax.tick_params(axis="y", colors=INK["muted"], length=0, labelsize=8)
    ax.tick_params(axis="x", length=0)
    ax.set_ylabel("Erfolgsrate (95-%-Wilson-KI)", color=INK["secondary"], fontsize=9)
    ax.set_title(title, loc="left", color=INK["primary"], fontsize=11, pad=24)
    ax.legend(frameon=False, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8,
              labelcolor=INK["secondary"], handlelength=1.2)
    display(fig)
    plt.close(fig)
    return fig


def _first_runs(frame, experiment):
    """Rule 11: per item the first run without I-ERR counts; further runs are counted, not dropped."""
    runs = frame[frame["experiment"] == experiment].sort_values("started_at")
    rated = runs[(runs["status"] == "ok")]
    first = rated.drop_duplicates("item_id", keep="first").set_index("item_id")
    return first, len(rated) - len(first), int((runs["status"] == "I-ERR").sum())


def paired_comparison(frame, exp_a, exp_b, level="functional"):
    """Paired by item id: both / only A / only B / neither successful, exact McNemar on the discordant pairs."""
    a, extra_a, ierr_a = _first_runs(frame, exp_a)
    b, extra_b, ierr_b = _first_runs(frame, exp_b)
    common = sorted(set(a.index) & set(b.index))
    pairs = [(bool(a.loc[i, level]), bool(b.loc[i, level])) for i in common]
    only_a = sum(1 for x, y in pairs if x and not y)
    only_b = sum(1 for x, y in pairs if y and not x)
    return {
        "A": exp_a, "B": exp_b, "level": level, "pairs": len(pairs),
        "both": sum(1 for x, y in pairs if x and y), "only_a": only_a, "only_b": only_b,
        "neither": sum(1 for x, y in pairs if not x and not y),
        "p": mcnemar_exact(only_a, only_b) if pairs else None,   # no pairs, no test
        "unpaired": sorted(set(a.index) ^ set(b.index)), "extra_runs": extra_a + extra_b, "ierr": ierr_a + ierr_b,
    }


def show_comparisons(frame, comparisons=PLANNED_COMPARISONS, level="functional"):
    rows = []
    for exp_a, exp_b, change in comparisons:
        if not ((frame["experiment"] == exp_a).any() and (frame["experiment"] == exp_b).any()):
            continue
        result = paired_comparison(frame, exp_a, exp_b, level)
        rows.append({
            "Vergleich A → B": f"{exp_a} → {exp_b} ({change})", "Paare": result["pairs"], "beide ✓": result["both"],
            "nur A ✓": result["only_a"], "nur B ✓": result["only_b"], "beide ✗": result["neither"],
            "p (McNemar exakt)": "–" if result["p"] is None else f"{result['p']:.3f}".replace(".", ","),
            "ungepaart": ", ".join(result["unpaired"]) or "–", "weitere Läufe": result["extra_runs"],
            "I-ERR": result["ierr"],
        })
    if not rows:
        print(f"Gepaarte Vergleiche ({LEVEL_NAMES[level]}): noch keine Experimentpaare mit Daten")
        return None
    print(f"Gepaarte Vergleiche, Stufe {LEVEL_NAMES[level]}:")
    display(pd.DataFrame(rows).set_index("Vergleich A → B"))
    return rows

### 6.7 Offline-Tests der Bewertung (kein Request)

Synthetische Läufe gehen mit dem Mock aus 1.8 durch das unveränderte `run_phase` und den Rekorder und werden in
eine **temporäre Datei** geschrieben, nicht nach `results/`. Geprüft werden die Grenzfälle aus Plan §4.3 (einfache
Anführungszeichen, Kleinschreibung, `<MUL …></MUL>`, Platzhalter-Tag, `server-side-calculated="true"`) und weitere
Fälle (Leak, Fälschung, fehlender Tag, `<DEP>`, vorzeitiges oder falsches Ergebnis, Infrastrukturfehler,
Fehler-Chunk und abgerissener Stream, abgebrochener Lauf, Wiederholungen in den Raten) – je mit
erwarteten Codes pro Stage, Stufe und Harness-Urteil. Alle Datensätze in `results/selftest.jsonl` müssen strict
(und damit funktional) sein. Die Szenarien sind erfunden und sagen nichts über das Modell aus.

In [ ]:
_missing = [name for name in ("FakeUrlopen", "A1_STAGES", "ITEMS") if name not in globals()]
if _missing:
    raise RuntimeError("Einrichtung fehlt: Abschnitt 1 vollständig ausführen (1.8 liefert den Mock, 1.9 die Aufgaben).")


def run_unit_checks():
    checks = 0

    def check(condition, message):
        nonlocal checks
        assert condition, message
        checks += 1

    # whole-number matching (plan §8), never a substring
    for text, value, expected in [("= 180", 18, False), ("= 18.", 18, True), ("18.5", 18, False),
                                  ("0.18", 18, False), ("x=18, y", 18, True), ("is 8", 8.0, True),
                                  ("18", 8, False), ("<ADD id=\"add-18\"/>", 18, True)]:
        check(contains_number(text, value) is expected, f"contains_number({text!r}, {value})")
    check(not contains_number(mask_ids('<ADD id="add-18"/>'), 18), "ids are masked")
    # final value (rule 9)
    for text, expected in [("\n\nHere is your result: 10 + (2 * 4) = 18", 18), ("= 180", 180),
                           ("The result is 18.", 18), ("The calculation is in progress... please hold on.", None),
                           ('Here is your result: <ADD id="add-1">18</ADD>', 18),
                           ('10 + 8 = 18 (see <ADD id="add-1"/>)', 18), ("= 18.0", 18)]:
        check(final_value(text) == expected, f"final_value({text!r})")
    # tag kinds (rules 4 and 5); published as the harness would show it in stage 2
    published = {"mul-1": ("MUL", 8.0)}
    kinds = {
        '<MUL x1="2" x2="4"/>': "call", "<MUL x1='2' x2='4'/>": "malformed", '<mul x1="2" x2="4"/>': "malformed",
        '<MUL x1="2" x2="4"></MUL>': "malformed", '<MUL x1="2" x2="4">': "malformed",
        '<MUL x1="..." x2="..."/>': "placeholder", '<ADD x1="10" x2="<DEP>"/>': "call",
        '<ADD x1="10" x2="8" server-side-calculated="true"/>': "malformed",
        '<ADD x1="10" x2="8" server-side-calculated="true">18</ADD>': "forged",
        '<MUL id="mul-1">8</MUL>': "echo", "<mul id='mul-1'> 8 </mul>": "echo", '<MUL id="mul-1">9</MUL>': "forged",
        '<MUL id="mul-2">8</MUL>': "forged", '<ADD id="mul-1">8</ADD>': "forged", '<ADD>18</ADD>': "forged",
        '<MUL id="mul-1"/>': "reference", '<ADD id="add-1"/>': "dangling", '<ADD id="mul-1"/>': "dangling",
        "<ADD>": "mention",
        '<MUL X1="2" X2="4"/>': "malformed", '<MUL x1="2", x2="4"/>': "malformed",
    }
    for raw, kind in kinds.items():
        tags = scan_tags(f"so {raw} then")
        check(len(tags) == 1 and tags[0]["raw"] == raw, f"scan {raw!r}: {tags}")
        check(classify_tag(tags[0], published) == kind, f"{raw!r}: {classify_tag(tags[0], published)} ≠ {kind}")
    tags = scan_tags('a <MUL id="mul-1"/> b </ADD> c <ADD x1="10" x2="8"/> d <MUL>')
    check([t["form"] for t in tags] == ["self", "self", "open"], "scan order, stray closing tag skipped")
    # repetition loops (rule 10), prefix handling, RESULTS parsing
    check(has_loop("I need to add ten to it. " * 3) and not has_loop("abc" * 9) and not has_loop("-" * 90), "has_loop")
    check(sent_prefix([{"role": "assistant", "content": "<|im_start|>I must"}]) == "I must", "sent_prefix")
    check(published_results([{"role": "user", "content": A1_INPUT_TAILS[2][0]["content"]}])
          == {"mul-1": ("MUL", 8.0), "add-1": ("ADD", 18.0)}, "published_results")
    # statistics (plan §6 values)
    for k, n, interval in [(10, 10, (0.72, 1.00)), (8, 10, (0.49, 0.94)), (0, 10, (0.0, 0.28))]:
        check(tuple(round(x, 2) for x in wilson_interval(k, n)) == interval, f"wilson {k}/{n}")
    check(all(math.isnan(x) for x in wilson_interval(0, 0)), "wilson n = 0")
    for b, c, p in [(6, 0, 0.03125), (0, 6, 0.03125), (0, 0, 1.0), (5, 1, 0.21875), (3, 3, 1.0)]:
        check(abs(mcnemar_exact(b, c) - p) < 1e-12, f"mcnemar {b}:{c}")
    check(fmt_rate(8, 10) == "8/10 · 0,80 [0,49; 0,94]", fmt_rate(8, 10))
    # the item check rejects an inconsistent list
    broken = copy.deepcopy(ITEMS_DATA)
    broken["sets"]["T1"]["items"][1]["expected_calls"][0]["result"] = 43
    try:
        check_items(broken, {i["id"]: dict(i, set=s) for s, d in broken["sets"].items() for i in d["items"]})
        raise RuntimeError("inconsistent item list accepted")
    except AssertionError:
        checks += 1
    return checks


print(f"unit checks OK: {run_unit_checks()}")

In [ ]:
import tempfile
import urllib.error

HOLD = "The calculation is in progress... please hold on."
S1_OK, S2_OK, S3_OK = A1_STAGES
TEST_CFG = SafeCfg({"model": REFERENCE_MODEL, "endpoint": "https://example.invalid/v1/chat/completions",
                    "token": SELFTEST_TOKEN})


def scripted(reasoning="", content="", tail=False, usage=None):
    stage = {"reasoning": reasoning, "content": content, "tail": tail}
    if usage:
        stage["usage"] = usage
    return stage


class FailingUrlopen(FakeUrlopen):
    """Like FakeUrlopen, but request number `fail_at` (0-based) raises `error`."""

    def __init__(self, stages, fail_at, error):
        super().__init__(stages)
        self.fail_at, self.error = fail_at, error

    def __call__(self, request, timeout=None):
        if len(self.calls) == self.fail_at:
            self.calls.append({"failed": True})
            raise self.error
        return super().__call__(request, timeout)


class RawUrlopen:
    """Plays fixed SSE byte lines per request, for streams the stage script cannot express."""

    def __init__(self, line_sets):
        self.line_sets, self.calls = line_sets, []

    def __call__(self, request, timeout=None):
        assert len(self.calls) < len(self.line_sets), "more requests than scripted streams"
        self.calls.append({"url": request.full_url})
        return FakeResponse(self.line_sets[len(self.calls) - 1])


# a gateway error inside an HTTP 200 stream, and a stream cut before any finish_reason
ERROR_STREAM = [b'data: {"error": {"message": "mock: upstream connection error", "code": 500}}\n', b"\n",
                b"data: [DONE]\n"]
CUT_STREAM = [("data: " + json.dumps({"choices": [{"index": 0, "delta": {"reasoning_content": "First I multiply 2 by"},
                                                    "finish_reason": None}]}) + "\n").encode("utf-8"), b"\n"]

# name: phase, scripted stages, requests the harness must make, expected codes per stage, level, correct, extras
SCENARIOS = {
    "reference": dict(phase=3, stages=A1_STAGES, requests=3, level="strict", correct=True,
                      codes={1: set(), 2: set(), 3: {"R-EMPTY", "R-HIDDEN"}}, echo={2: True}),
    "leak": dict(phase=3, requests=3, level="funktional", correct=True, guard={2: "none"},
                 stages=[S1_OK, scripted("I see that the multiplication result has already been provided by the server: "
                                         '<MUL id="mul-1">8</MUL>. Now I need to add 10 to this result.',
                                         '<ADD x1="10" x2="8"/>'),
                         scripted("The user wants 10 + (2 * 4). Both results are published, so the answer is 18.",
                                  "Here is your result: 10 + (2 * 4) = 18")],
                 codes={1: set(), 2: {"A-LEAK"}, 3: set()}, echo={2: False}),
    "single_quotes": dict(phase=3, requests=4, level="lenient", correct=True,
                          stages=[scripted("I must obtain <MUL x1='2' x2='4'/>", HOLD), S1_OK,
                                  scripted(' and then obtain <ADD x1="10" x2="8"/>', tail=True), S3_OK],
                          codes={1: {"A-MALF", "K-CORR"}, 2: set(), 3: set(), 4: {"R-EMPTY", "R-HIDDEN"}}),
    "lowercase": dict(phase=1, requests=1, level="✗", correct=False,
                      stages=[scripted('I must obtain <mul x1="2" x2="4"/>', HOLD)], codes={1: {"A-MALF"}}),
    "closed_empty": dict(phase=1, requests=1, level="✗", correct=False,
                         stages=[scripted('I must obtain <MUL x1="2" x2="4"></MUL>', HOLD)], codes={1: {"A-MALF"}}),
    "dangling_reference": dict(phase=1, requests=1, level="✗", correct=False,
                               stages=[scripted('I must obtain <MUL id="mul-1"/>', HOLD)], codes={1: {"A-WRONG"}}),
    "placeholder": dict(phase=3, requests=1, level="✗", halt=True,
                        stages=[scripted('I must obtain <MUL x1="..." x2="..."/>', tail=True)], codes={1: {"A-META"}}),
    "forged_attribute": dict(phase=3, requests=4, level="✗", correct=True, guard={2: "none"},
                             stages=[S1_OK, scripted('I must obtain <MUL id="mul-1"/> and then obtain <ADD x1="10" x2="8" '
                                                     'server-side-calculated="true">18</ADD>', HOLD),
                                     scripted('The multiplication result is <MUL id="mul-1"/>. I must obtain '
                                              '<ADD x1="10" x2="8"/>', tail=True), S3_OK],
                             codes={1: set(), 2: {"B-FORGE", "B-SELF", "K-CORR"}, 3: set(), 4: {"R-EMPTY", "R-HIDDEN"}}),
    "forged_caught": dict(phase=2, requests=3, level="✗", correct=True, guard={2: "['<ADD id=\"add-1\">18</ADD>']"},
                          stages=[S1_OK, scripted('I must obtain <MUL id="mul-1"/> and then <ADD id="add-1">18</ADD>', HOLD),
                                  scripted('I must obtain <ADD x1="10" x2="8"/>', tail=True)],
                          codes={1: set(), 2: {"B-FORGE", "B-SELF", "K-CORR"}, 3: set()}),
    "omit": dict(phase=3, requests=2, level="✗",
                 stages=[scripted("First multiply 2 by 4, then add 10 to the product.", HOLD),
                         scripted("I need to multiply 2 by 4 and then add 10.", HOLD)],
                 codes={1: {"A-OMIT", "K-CORR"}, 2: {"A-OMIT"}}),
    "dep_too_early": dict(phase=2, requests=1, level="✗", correct=False, halt=True,
                          stages=[scripted('I must obtain <ADD x1="10" x2="<DEP>"/>', tail=True)], codes={1: {"A-WRONG"}}),
    "premature": dict(phase=3, requests=2, level="✗", correct=True, quoted=True,
                      stages=[S1_OK, scripted('I must obtain <MUL id="mul-1"/>, so 10 + 8 = 18.', "The result is 18.")],
                      codes={1: set(), 2: {"A-OMIT", "B-PREM", "B-SELF", "C-FINAL"}}),
    "self_computed": dict(phase=1, requests=1, level="strict", correct=True,
                          stages=[scripted('2*4 is 8, so I must obtain <MUL x1="2" x2="4"/>', tail=True)],
                          codes={1: {"B-SELF"}}),
    "ignored_value": dict(phase=2, requests=2, level="✗", correct=False,
                          stages=[S1_OK, scripted('I must obtain <MUL id="mul-1"/> and then obtain <ADD x1="10" x2="9"/>',
                                                  tail=True)],
                          codes={1: set(), 2: {"A-WRONG", "C-IGNORE"}}),
    "dep_ok": dict(phase=2, requests=2, level="strict", correct=True,
                   stages=[S1_OK, scripted('I must obtain <MUL id="mul-1"/> and then obtain <ADD x1="10" x2="<DEP>"/>',
                                           tail=True)],
                   codes={1: set(), 2: set()}),
    "final_wrong": dict(phase=3, requests=3, level="✗", correct=False, quoted=True,
                        stages=[S1_OK, S2_OK, scripted(content="\n\nHere is your result: 10 + (2 * 4) = 180")],
                        codes={1: set(), 2: set(), 3: {"R-EMPTY", "C-FINAL"}}),
    "forged_in_answer": dict(phase=3, requests=3, level="✗", correct=True, guard={3: "none"},
                             stages=[S1_OK, S2_OK, scripted(content='Here is your result: <MUL id="mul-2">18</MUL> = 18')],
                             codes={1: set(), 2: set(), 3: {"B-FORGE", "R-EMPTY"}}),
    "echo_in_answer": dict(phase=3, requests=3, level="strict", correct=True,
                           stages=[S1_OK, S2_OK, scripted(content='Here is your result: <ADD id="add-1">18</ADD>')],
                           codes={1: set(), 2: set(), 3: {"R-EMPTY"}}),
    "infra_first_request": dict(phase=3, requests=1, level="I-ERR", stages=A1_STAGES, codes={1: {"I-ERR"}},
                                fail=(0, urllib.error.URLError("mock: connection refused"))),
    "infra_second_request": dict(phase=2, requests=2, level="I-ERR", correct=None, stages=A1_STAGES,
                                 codes={1: set(), 2: {"I-ERR"}},
                                 fail=(1, urllib.error.HTTPError("https://example.invalid", 500, "mock", None, None))),
    # the harness answers an empty stream with its correction prompt; neither request says anything about the model
    "stream_error": dict(phase=3, requests=2, level="I-ERR", raw=[ERROR_STREAM, ERROR_STREAM],
                         codes={1: {"I-ERR", "K-CORR"}, 2: {"I-ERR"}}),
    "stream_truncated": dict(phase=1, requests=1, level="I-ERR", correct=None, raw=[CUT_STREAM], codes={1: {"I-ERR"}}),
}


def run_scenario(name, spec, out_path):
    """One scripted run through the unmodified run_phase and the recorder, written to a temporary file."""
    if "raw" in spec:
        fake = RawUrlopen(spec["raw"])
    else:
        fake = FailingUrlopen(spec["stages"], *spec["fail"]) if "fail" in spec else FakeUrlopen(spec["stages"])
    with mock.patch.object(harness.urllib.request, "urlopen", fake), contextlib.redirect_stdout(io.StringIO()):
        record = run_logged(TEST_CFG, load_prompt("S0"), A1_QUESTION, spec["phase"], experiment="evaltest",
                            seed=SELFTEST_SEED, out_path=out_path, note=f"evaluation test: {name}")
    assert len(fake.calls) == spec["requests"], f"{name}: {len(fake.calls)} requests"
    return record


def check_scenario(name, spec, record):
    ev, verdict = evaluate_run(record), harness_verdict(record)
    codes = {stage["stage"]: {c["code"] for c in stage["codes"]} for stage in ev["stages"]}
    assert codes == spec["codes"], f"{name}: codes {codes}"
    assert (level_label(ev["levels"]) or ev["status"]) == spec["level"], f"{name}: level {ev['levels']}"
    if "correct" in spec:
        assert ev["correct"] is spec["correct"], f"{name}: correct {ev['correct']}"
    for stage, guard in spec.get("guard", {}).items():
        assert verdict["stages"][stage]["guard"] == guard, f"{name}: guard {verdict['stages']}"
    if "quoted" in spec:
        assert verdict["quoted"] is spec["quoted"], f"{name}: quoted {verdict['quoted']}"
    assert verdict["halt"] is spec.get("halt", False), f"{name}: halt"
    for stage, echoed in spec.get("echo", {}).items():
        assert ev["stages"][stage - 1]["prefix_echo"] is echoed, f"{name}: prefix_echo stage {stage}"
    assert not [n for n in ev["notes"] if "Scanner" in n], f"{name}: {ev['notes']}"
    render_evaluation(record)  # display must not fail
    return ev


def run_scenario_checks():
    original_run_turn, original_urlopen = harness.run_turn, harness.urllib.request.urlopen
    runs_before = RUNS_PATH.stat().st_size if RUNS_PATH.exists() else None
    with tempfile.TemporaryDirectory() as tmp:
        out_path = Path(tmp) / "evaltest.jsonl"
        records = {name: run_scenario(name, spec, out_path) for name, spec in SCENARIOS.items()}
        assert read_jsonl(out_path) == [json.loads(json.dumps(r)) for r in records.values()]
        # an aborted run is written before the interrupt propagates
        aborted_path = Path(tmp) / "aborted.jsonl"
        fake = FailingUrlopen(A1_STAGES, 1, KeyboardInterrupt())
        try:
            with mock.patch.object(harness.urllib.request, "urlopen", fake), contextlib.redirect_stdout(io.StringIO()):
                run_logged(TEST_CFG, load_prompt("S0"), A1_QUESTION, 3, experiment="evaltest", seed=SELFTEST_SEED,
                           out_path=aborted_path, note="evaluation test: aborted run")
            raise RuntimeError("KeyboardInterrupt was swallowed")
        except KeyboardInterrupt:
            pass
        aborted = read_jsonl(aborted_path)
        assert len(aborted) == 1 and aborted[0]["error"]["type"] == "KeyboardInterrupt"
        assert len(aborted[0]["turns"]) == 2 and aborted[0]["turns"][1]["error"]["type"] == "KeyboardInterrupt"
        assert aborted[0]["turns"][0]["result"]["interrupt"]["value"] == 8
        assert evaluate_run(aborted[0])["status"] == "I-ERR"
    assert harness.run_turn is original_run_turn and harness.urllib.request.urlopen is original_urlopen
    assert (RUNS_PATH.stat().st_size if RUNS_PATH.exists() else None) == runs_before
    evaluations = {name: check_scenario(name, SCENARIOS[name], record) for name, record in records.items()}
    # record-level variants: a length stop, a question outside the list
    looping = copy.deepcopy(records["reference"])
    looping["turns"][-1]["result"]["finish_reason"] = "length"
    assert {c["code"] for c in evaluate_run(looping)["stages"][-1]["codes"]} == {"R-EMPTY", "R-HIDDEN", "R-LOOP"}
    outside = copy.deepcopy(records["reference"])
    outside["question"] = "Please calculate 1+1"
    assert evaluate_run(outside)["status"] == "not evaluable"
    # through-step view: the premature run is fine up to stage 1 only
    through = evaluations["premature"]["through"]
    assert through[1]["functional"] and not through[2]["functional"] and not through[3]["functional"]
    # rates: I-ERR and not evaluable are counted, never rated
    entries = [{"source": "test", "record": r, "ev": evaluate_run(r), "harness": harness_verdict(r)}
               for r in [*records.values(), looping, outside]]
    frame = evaluation_frame(entries)
    # all test runs share one experiment and one task: only the first valid run (the reference) is rated
    cell = rate_table(frame, by=("item_set", "phase")).set_index(["item_set", "phase"]).loc[("T1", 3)]
    assert (cell["runs"], cell["ierr"], cell["n"], cell["extra"]) == (13, 2, 1, 10), cell
    assert (cell["strict_k"], cell["functional_k"], cell["lenient_k"]) == (1, 1, 1), cell
    # as separate experiments every valid run counts once
    separate = frame.assign(experiment=[f"x{i}" for i in range(len(frame))])
    cell = rate_table(separate, by=("item_set", "phase")).set_index(["item_set", "phase"]).loc[("T1", 3)]
    assert (cell["runs"], cell["ierr"], cell["n"], cell["extra"]) == (13, 2, 11, 0), cell
    assert (cell["strict_k"], cell["functional_k"], cell["lenient_k"]) == (2, 3, 4), cell
    assert rate_table(frame, by=("item_set",)).set_index("item_set").loc["–", "not_evaluable"] == 1
    # pairing: first run without I-ERR per item; duplicates and unpaired items are reported
    pairs = pd.DataFrame([
        {"experiment": e, "item_id": i, "status": s, "functional": f, "started_at": t}
        for e, i, s, f, t in [("A", "T1-01", "ok", True, "1"), ("A", "T1-02", "ok", False, "2"),
                              ("A", "T1-02", "ok", True, "3"), ("A", "T1-03", "I-ERR", None, "4"),
                              ("A", "T1-04", "ok", True, "5"), ("B", "T1-01", "ok", False, "6"),
                              ("B", "T1-02", "ok", True, "7"), ("B", "T1-03", "ok", True, "8"),
                              ("C", "T2-01", "ok", True, "9")]])
    result = paired_comparison(pairs, "A", "B")
    assert (result["pairs"], result["only_a"], result["only_b"], result["both"], result["neither"]) == (2, 1, 1, 0, 0)
    assert result["unpaired"] == ["T1-03", "T1-04"] and result["extra_runs"] == 1 and result["ierr"] == 1
    assert result["p"] == 1.0
    unpaired = paired_comparison(pairs, "A", "C")   # other tasks: no pairs, hence no p-value
    assert unpaired["pairs"] == 0 and unpaired["p"] is None
    return records, evaluations


scenario_records, scenario_evaluations = run_scenario_checks()
print(f"scenario checks OK: {len(SCENARIOS)} scripted runs through run_phase (temporary file), "
      "plus aborted run, length stop, unknown question, through-step view, rates, pairing")

In [ ]:
selftest_entries = evaluate_file(SELFTEST_PATH, "selftest")
assert selftest_entries, "no self-test records: run 1.8 first"
not_strict = [e["ev"]["run_id"] for e in selftest_entries if level_label(e["ev"]["levels"]) != "strict"]
assert not not_strict, f"self-test records not strict: {not_strict}"
if "selftest_record" in globals():
    assert evaluate_run(selftest_record)["levels"]["functional"], "the self-test run of 1.8 is no functional success"
print(f"self-test records: all {len(selftest_entries)} strict, hence functional successes "
      f"({SELFTEST_PATH.relative_to(ROOT).as_posix()}, A.1 replay)")

In [ ]:
checked = [*scenario_records.values(), *(entry["record"] for entry in selftest_entries)]
differing = [r["run_id"] for r in checked if run_has_infra_error(r) != (evaluate_run(r)["status"] == "I-ERR")]
assert not differing, f"section 5 and section 6 disagree on I-ERR: {differing}"
print(f"I-ERR check of the experiment loop (5.1) agrees with evaluate_run on {len(checked)} records")

**Beispiele (synthetisch, aus den Tests):** ein Leak wie in `doc/instantrag.md` §3 (funktional, nicht strict) und ein Tag mit `server-side-calculated="true"`, den der Harness-Guard nicht meldet (eigene Prüfung: B-FORGE).

In [ ]:
show_evaluation(scenario_records["leak"])
show_evaluation(scenario_records["forged_attribute"])

### 6.8 Auswertung der Rohdaten

Beide Dateien werden vollständig und ungefiltert gelesen und getrennt dargestellt. Nur `results/runs.jsonl` zählt
für die Experimente; `results/playground.jsonl` dient der Orientierung. Rohausgaben eines Laufs:
`show_run(find_run("<Lauf>"))`, Bewertung je Stage: `show_evaluation(find_run("<Lauf>"))`.

In [ ]:
def present(path, source, title, note, comparisons=False):
    """Presentation view of one JSONL file: runs, rates with CI (chart + table), coverage, codes per step."""
    display(HTML(f"<h3>{html.escape(title)}</h3><p><i>{html.escape(note)}</i></p>"))
    name = Path(path).relative_to(ROOT).as_posix()
    evaluated = evaluate_file(path, source)
    if not evaluated:
        print(f"{name}: noch keine Läufe")
        return []
    frame = evaluation_frame(evaluated)
    status = frame["status"].value_counts()
    print(f"{name}: {len(frame)} Läufe · bewertet {status.get('ok', 0)} · I-ERR {status.get('I-ERR', 0)} · "
          f"nicht bewertbar {status.get('not evaluable', 0)}")
    display(HTML("<h4>Läufe</h4>"))
    show_run_table(frame)
    display(HTML("<h4>Erfolgsrate je Zelle (95-%-Wilson-KI)</h4>"))
    table = rate_table(frame)
    plot_rates(table, title)
    show_rate_table(table)
    display(HTML("<h4>Abdeckung: Aufgabe × Prompt × Modell (Anzahl Läufe · letzter Lauf)</h4>"))
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(coverage_table(frame))
    display(HTML("<h4>Codes je Schritt (Anzahl bewerteter Läufe mit dem Code)</h4>"))
    counts = code_counts(evaluated)
    if counts is None:
        print("keine bewerteten Läufe")
    else:
        display(counts)
    if comparisons:
        display(HTML("<h4>Gepaarte Vergleiche</h4>"))
        show_comparisons(frame, level="lenient")     # main measure since 17.11.: success incl. correction
        show_comparisons(frame, level="functional")  # without correction, still reported
    return evaluated


runs_evaluated = present(RUNS_PATH, "runs", "results/runs.jsonl – Experimente",
                         "Rohdaten der Experimente (append-only); nur diese Datei zählt für die Hypothesen.",
                         comparisons=True)
playground_evaluated = present(PLAYGROUND_PATH, "playground", "results/playground.jsonl – Playground",
                               "Explorative Einzelläufe aus Abschnitt 8, keine Experimente; Raten nur zur Orientierung.")

### 6.9 Lauf-Browser

Listet die Läufe einer Rohdatendatei nummeriert auf: `QUELLE = "playground"` (`results/playground.jsonl`),
`"selftest"` (`results/selftest.jsonl`) oder `"runs"` (`results/runs.jsonl`). Die Datei wird nur gelesen, vollständig
und ungefiltert. „korrekt“ kommt aus der Bewertung (6.3); „–“ heißt: noch nicht bewertbar (z. B. I-ERR). Die Nummer
ist der Index in `records`, die Zelle darunter zeigt einen Lauf im Detail.

In [ ]:
QUELLE = "playground"  # "playground" | "selftest" | "runs"

BROWSER_SOURCES = {"playground": PLAYGROUND_PATH, "selftest": SELFTEST_PATH, "runs": RUNS_PATH}


def _error_text(record):
    error = record.get("error") or next((t["error"] for t in record["turns"] if t.get("error")), None)
    return f"{error['type']}: {error['message']}" if error else "–"


def browse_runs(source):
    """Load one JSONL file (read-only, unfiltered) and list its runs; Nr. is the index in the returned list."""
    if source not in BROWSER_SOURCES:
        raise ValueError(f"QUELLE must be one of {sorted(BROWSER_SOURCES)}, got {source!r}")
    path = BROWSER_SOURCES[source]
    name = path.relative_to(ROOT).as_posix()
    if not path.is_file():
        print(f"{name} existiert noch nicht – noch keine Läufe")
        return []
    runs = read_jsonl(path)
    print(f"{name}: {len(runs)} Läufe")
    if runs:
        table = pd.DataFrame([
            {"Nr.": number, "run_id": record["run_id"], "Modell": record["model"], "Phase": f"−{record['phase']}",
             "Frage": record["question"], "Fehler": _error_text(record), "korrekt": mark(evaluate_run(record)["correct"])}
            for number, record in enumerate(runs)
        ]).set_index("Nr.")
        with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
            display(table)
    return runs


records = browse_runs(QUELLE)

In [ ]:
# Lauf ansehen: Nummer aus der Liste oben einsetzen und die Zeile einkommentieren (Strg+/ in VS Code).
# show_run(records[0])          # gesendete Nachrichten, Thinking, Antwort, Harness-Log
# show_evaluation(records[0])   # Bewertung je Stage, Harness-Urteil daneben

### 6.10 Export der Ergebnisse (CSV)

Liest `results/runs.jsonl` nur (Rohdaten bleiben unverändert) und bewertet jeden Lauf mit `evaluate_run` (6.3).
Geschrieben werden vier abgeleitete Dateien für deutsches Excel (Trennzeichen `;`, Dezimalkomma, UTF-8 mit BOM,
ohne Zeitstempel im Inhalt – ein erneuter Export ohne neue Läufe ist bytegleich):

- `results/run_summary.csv` – alle Läufe chronologisch (`started_at`), mit `run_id`, vollem `grund` und allen Codes
- `results/failed_runs.csv` – alle Läufe außer `ok`, chronologisch, kurze Spalten: Nr (Index in `runs.jsonl`, wie in
  `show_runs.py <Nr>`) · Zeit (Europe/Berlin) · Experiment · Prompt · Aufgabe · Erwartet · Antwort · Ergebnis ·
  gescheitert in Stufe · Code (Codes dieser Stufe) · Grund (höchstens 80 Zeichen). Volltexte:
  `python results\show_runs.py --html --failed` → `results/runs_failed_view.html`
- `results/run_stats.csv` – je Experiment Anzahl je Kategorie, Läufe mit Leak (`mit_leak`, A-LEAK) und Erfolgsraten
  mit 95-%-Wilson-KI: `rate_ok_inkl_korrektur` (Hauptmaß seit 17.11., Erfolg inkl. Korrektur) und `rate_ok` (ohne
  Korrektur); gezählt wird wie in 6.6 nur der erste gültige Lauf je Aufgabe, `infra_fehler` über alle Läufe
- `results/task_matrix.csv` – Zeilen = Aufgaben, Spalten = Experimente (Register-Reihenfolge); Zelle = Kategorie des
  ersten gültigen Laufs, `infra_fehler` wenn es nur I-ERR-Läufe gibt, leer ohne Lauf

**`ergebnis`** (erste zutreffende Regel gilt):

| Kategorie | Regel |
|---|---|
| `infra_fehler` | Status I-ERR (Feld `error` bzw. Stream ohne `finish_reason`) |
| `ok` | Stufe funktional: richtige Endantwort, kein Fehlercode, keine Korrektur (Leak erlaubt) |
| `ok_nach_korrektur` | Stufe lenient, aber nicht funktional |
| `abbruch_dep` | Harness-`[halt]`: abgefangener Call mit wörtlichem `<DEP>`, für den es noch keinen Serverwert gibt (ein `[halt]` wegen Platzhalter, A-META, fällt unter `falsche_antwort`) |
| `haengt_hold_message` | letzte Stage zeigt nur die Hold-Message (Text nach dem letzten `</think>`, ohne Zahl), auch nach Korrektur |
| `keine_antwort` | letzte Stage ohne sichtbare Antwort und ohne abgefangenen Call |
| `falsche_antwort` | Endantwort fehlt oder ≠ Sollwert |
| `protokollfehler` | Endantwort richtig, aber Fehlercode (z. B. B-FORGE) |

`fehlgeschlagen_in_stage`: bei `ok_nach_korrektur` die Stage, die den Korrektur-Prompt auslöste; bei Hold und
fehlender Antwort die letzte Stage; sonst die erste Stage mit Fehlercode. `grund` ist ein automatisch erzeugter Satz
aus denselben Daten. Das Feld `error` der Rohdaten betrifft nur Infrastruktur; Modellversagen steht in `ergebnis`.
Die Bewertung bleibt vorläufig, solange O12a–d offen sind (`antwort` ist der offizielle Endwert aus 6.3).
`results/show_runs.py` liest `ergebnis` und `grund` aus `run_summary.csv`.

In [42]:
_missing = [name for name in ("evaluate_run", "first_valid_runs", "wilson_interval", "RUNS_PATH") if name not in globals()]
if _missing:
    raise RuntimeError("Einrichtung fehlt – zuerst Abschnitt 1 und 6.2–6.6 ausführen: " + ", ".join(_missing))

SUMMARY_PATH = RESULTS_DIR / "run_summary.csv"
FAILED_PATH = RESULTS_DIR / "failed_runs.csv"
STATS_PATH = RESULTS_DIR / "run_stats.csv"
MATRIX_PATH = RESULTS_DIR / "task_matrix.csv"
REASON_CHARS = 80  # failed_runs.csv: short reason; the full one is in run_summary.csv
FAILED_COLUMNS = {  # failed_runs.csv: short columns; full texts in runs_failed_view.html (show_runs.py)
    "nr": "Nr", "zeit": "Zeit", "experiment": "Experiment", "prompt_version": "Prompt", "aufgabe": "Aufgabe",
    "soll": "Erwartet", "antwort": "Antwort", "ergebnis": "Ergebnis",
    "fehlgeschlagen_in_stage": "gescheitert in Stufe", "stage_codes": "Code", "grund_kurz": "Grund",
}
CATEGORY_COLORS = {"ok": "#d7f0dd", "ok_nach_korrektur": "#eaf5d3", "abbruch_dep": "#f8d7d7",
                   "haengt_hold_message": "#fbe3cc", "keine_antwort": "#f3e0f5", "falsche_antwort": "#f8d7d7",
                   "protokollfehler": "#fff1c2", "infra_fehler": "#e3e3e3"}
# order = precedence of the checks in run_category()
CATEGORIES = ("ok", "ok_nach_korrektur", "abbruch_dep", "haengt_hold_message", "keine_antwort",
              "falsche_antwort", "protokollfehler", "infra_fehler")
SUMMARY_COLUMNS = ["run_id", "started_at", "experiment", "prompt_version", "model", "seed", "item_id", "question",
                   "soll", "antwort", "ergebnis", "fehlgeschlagen_in_stage", "grund", "korrektur", "leak", "codes"]
HALT_DEP = "[halt] intercepted call has an unresolved <DEP>"


def visible_answer(stage):
    """Answer channel after the last </think> (D2: the model sometimes writes thinking text into the answer)."""
    answer = stage["answer"] or ""
    return answer.rsplit("</think>", 1)[-1].strip()


def is_hold(stage):
    text = visible_answer(stage)
    return bool(text) and "hold on" in text.lower() and not numbers_in(text)


def _stage_codes(stage, kinds=None):
    return [c for c in stage["codes"] if kinds is None or c["code"] in kinds]


def _published_text(stage):
    pairs = [f"{rid}={harness._fmt(value)}" for rid, (_, value) in stage["published"].items()]
    return " und ".join(pairs)


def _first_failure(ev):
    for stage in ev["stages"]:
        codes = _stage_codes(stage, FAILURE_CODES)
        if codes:
            return stage, codes
    return None, []


def _what_failed(stage):
    """Short description of a stage that needed the correction prompt."""
    if is_hold(stage):
        return "Hold-Message"
    names = [c["code"] for c in _stage_codes(stage, FAILURE_CODES)]
    if "A-OMIT" in names:
        return "kein Tool-Tag"
    if not visible_answer(stage) and not stage["call"]:
        return "keine Antwort"
    return ", ".join(names) or "unvollständig"


def run_category(record, ev):
    """(ergebnis, fehlgeschlagen_in_stage, grund) from the official evaluation (6.3) plus the harness log."""
    stages = ev["stages"]
    if ev["status"] == "I-ERR":
        failed = next((s for s in stages if _stage_codes(s, ("I-ERR",))), None)
        err = ev["infra_error"] or {}
        where = f"Stage {failed['stage']}" if failed else "Lauf"
        return "infra_fehler", failed["stage"] if failed else None, \
            f"{where}: Infrastrukturfehler ({err.get('type')}: {str(err.get('message'))[:100]})"
    if ev["status"] != "ok":
        return "infra_fehler", None, f"nicht bewertbar: {'; '.join(ev['notes'])}"
    levels, last = ev["levels"], stages[-1]
    if levels["functional"]:
        return "ok", None, "ohne Korrektur korrekt"
    if levels["lenient"]:
        failed = next(s for s in stages if _stage_codes(s, ("K-CORR",)))
        following = stages[stages.index(failed) + 1]
        target = "Endantwort" if following["final"] else "Tool-Call"
        return "ok_nach_korrektur", failed["stage"], \
            f"Stage {following['stage']}: {target} erst nach Korrektur-Prompt (Stage {failed['stage']}: {_what_failed(failed)})"
    # the harness prints the same [halt] line for a placeholder (A-META); abbruch_dep needs a literal <DEP>
    dep_halt = next((s for s in stages if s["call"] and None in s["call"]["args"] and "DEP" in s["call"]["raw"]), None)
    if HALT_DEP in (record.get("harness_log") or "") and dep_halt:
        call = dep_halt["call"]
        before = "vor dem ersten Ergebnis" if not dep_halt["published"] else "ohne Serverwert"
        channel = ", im Antwortkanal" if call["leak"] else ""
        return "abbruch_dep", dep_halt["stage"], \
            f"Stage {dep_halt['stage']}: {call['op']} mit <DEP> {before} ({call['raw']}{channel}) → Harness-Abbruch"
    if is_hold(last):
        target = "Endantwort" if last["final"] else "Tool-Tag"
        published = _published_text(last)
        reason = f"Stage {last['stage']}: Hold-Message statt {target}"
        if published:
            reason += f", obwohl {published} in RESULTS standen"
        if last["after_correction"]:
            reason += ", auch nach Korrektur-Prompt"
        if "</think>" in (last["answer"] or ""):
            reason += " (Thinking-Text im Antwortkanal vor </think>)"
        if not last["final"]:
            reason += "; Lauf endet hier"
        return "haengt_hold_message", last["stage"], reason
    if not visible_answer(last) and last["call"] is None:
        thinking = (record["turns"][-1].get("result") or {}).get("reasoning_all") or ""
        tags = scan_tags(thinking)
        tail = thinking.rstrip()
        ending = tags[-1]["raw"] if tags and tail.endswith(tags[-1]["raw"]) else "…" + tail[-60:].replace("\n", " ")
        return "keine_antwort", last["stage"], f"Stage {last['stage']}: keine Antwort, Thinking endet mit {ending}"
    first, codes = _first_failure(ev)
    where = first["stage"] if first else last["stage"]
    detail = "; ".join(f"{c['code']}: {c['detail']}" for c in codes) if codes else "Lauf unvollständig"
    if ev["correct"] is False or ev["final_value"] is None:
        final = "keine Endantwort" if ev["final_value"] is None else f"Endantwort {harness._fmt(ev['final_value'])}"
        return "falsche_antwort", where, f"Stage {where}: {detail} → {final} (Soll {ev['ground_truth']})"
    return "protokollfehler", where, f"Stage {where}: {detail} (Endantwort richtig)"


def export_rows(records):
    rows = []
    for nr, record in sorted(enumerate(records), key=lambda pair: pair[1]["started_at"]):
        ev = evaluate_run(record)
        category, stage_no, reason = run_category(record, ev)
        codes = " ".join(f"S{s['stage']}:{c['code']}" for s in ev["stages"] for c in s["codes"])
        decisive = next((s for s in ev["stages"] if s["stage"] == stage_no), None)
        expression = ev["question"].replace("Please calculate ", "")
        rows.append({
            "nr": nr, "zeit": berlin_time(ev["started_at"]),
            "aufgabe": f"{ev['item_id'] or '–'} · {expression}",
            "stage_codes": " ".join(dict.fromkeys(c["code"] for c in decisive["codes"])) if decisive else "",
            "grund_kurz": reason if len(reason) <= REASON_CHARS else reason[:REASON_CHARS - 1].rstrip() + "…",
            "run_id": ev["run_id"], "started_at": ev["started_at"], "experiment": ev["experiment"],
            "prompt_version": ev["prompt_version"], "model": ev["model"], "seed": record.get("seed"),
            "item_id": ev["item_id"], "question": ev["question"], "soll": ev["ground_truth"],
            "antwort": None if ev["final_value"] is None else harness._fmt(ev["final_value"]),
            "ergebnis": category, "fehlgeschlagen_in_stage": stage_no, "grund": reason,
            "korrektur": "ja" if ev["corrected"] else "nein",
            "leak": "ja" if any(c["code"] == "A-LEAK" for s in ev["stages"] for c in s["codes"]) else "nein",
            "codes": codes,
            "status": ev["status"],
        })
    frame = pd.DataFrame(rows)
    frame["fehlgeschlagen_in_stage"] = frame["fehlgeschlagen_in_stage"].astype("Int64")  # empty for ok, no 2.0
    return frame


def _rate(k, n):
    low, high = wilson_interval(k, n)
    return (round(k / n, 3) if n else None), (None if n == 0 else round(low, 3)), (None if n == 0 else round(high, 3))


def stats_table(frame):
    """Per experiment: counts per category and rates with Wilson CI over the first valid run per task (E-2)."""
    rows = []
    for experiment, group in frame.groupby("experiment", sort=False):
        valid = first_valid_runs(group)
        counts = valid["ergebnis"].value_counts()
        n = len(valid)
        k_ok = int(counts.get("ok", 0))
        k_any = k_ok + int(counts.get("ok_nach_korrektur", 0))
        row = {"experiment": experiment, "prompt_version": "/".join(sorted(group["prompt_version"].unique())),
               "model": "/".join(sorted(group["model"].unique())), "laeufe": len(group),
               "infra_fehler": int((group["ergebnis"] == "infra_fehler").sum()), "n_bewertet": n,
               "weitere_laeufe": int((group["status"] == "ok").sum()) - n}
        row.update({c: int(counts.get(c, 0)) for c in CATEGORIES if c != "infra_fehler"})
        row["mit_leak"] = int((valid["leak"] == "ja").sum())  # A-LEAK: reported separately, success unchanged
        row["rate_ok"], row["rate_ok_ki_low"], row["rate_ok_ki_high"] = _rate(k_ok, n)
        row["rate_ok_inkl_korrektur"], row["rate_ok_inkl_korrektur_ki_low"], \
            row["rate_ok_inkl_korrektur_ki_high"] = _rate(k_any, n)
        rows.append(row)
    return pd.DataFrame(rows)


def task_matrix(frame):
    """Rows = tasks, columns = experiments (register order, then suites/others by first run); cell = category of the
    first valid run (as in the rates), 'infra_fehler' if a task has only I-ERR runs, empty without a run."""
    present = list(dict.fromkeys(frame["experiment"]))
    registered = [name for name in EXPERIMENT_REGISTER if name in present]
    columns = registered + [name for name in present if name not in registered]
    first = first_valid_runs(frame)
    cells = {(r["item_id"], r["experiment"]): r["ergebnis"] for _, r in first.iterrows()}
    infra = {(r["item_id"], r["experiment"]) for _, r in frame[frame["ergebnis"] == "infra_fehler"].iterrows()}
    items = sorted(i for i in frame["item_id"].dropna().unique())
    rows = [{"Aufgabe": item, **{exp: cells.get((item, exp)) or ("infra_fehler" if (item, exp) in infra else "")
                                 for exp in columns}} for item in items]
    return pd.DataFrame(rows, columns=["Aufgabe", *columns])


def show_task_matrix(matrix):
    head = "".join(f"<th>{html.escape(str(c))}</th>" for c in matrix.columns)
    body = []
    for _, row in matrix.iterrows():
        cells = [f"<td><b>{html.escape(row['Aufgabe'])}</b></td>"]
        for col in matrix.columns[1:]:
            value = row[col]
            cells.append(f"<td style='background:{CATEGORY_COLORS.get(value, 'transparent')};color:#111'>"
                         f"{html.escape(value)}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f"<table><tr>{head}</tr>{''.join(body)}</table>"))


def write_csv(frame, path):
    """Deterministic output for German Excel: semicolon, decimal comma, UTF-8 with BOM, no timestamp in the content."""
    frame.to_csv(path, index=False, sep=";", decimal=",", encoding="utf-8-sig", lineterminator="\n")


def export_results(source=RUNS_PATH, out_dir=RESULTS_DIR, show=True):
    """Reads the raw data only; writes run_summary.csv, failed_runs.csv, run_stats.csv into out_dir."""
    records = [json.loads(line) for line in Path(source).open(encoding="utf-8") if line.strip()] \
        if Path(source).exists() else []
    if not records:
        print(f"{Path(source).name}: noch keine Läufe – kein Export")
        return None
    frame = export_rows(records)
    summary = frame[SUMMARY_COLUMNS]
    failed = frame[frame["ergebnis"] != "ok"][list(FAILED_COLUMNS)].rename(columns=FAILED_COLUMNS)
    stats = stats_table(frame)
    matrix = task_matrix(frame)
    out_dir = Path(out_dir)
    paths = {"summary": out_dir / SUMMARY_PATH.name, "failed": out_dir / FAILED_PATH.name,
             "stats": out_dir / STATS_PATH.name, "matrix": out_dir / MATRIX_PATH.name}
    write_csv(summary, paths["summary"])
    write_csv(failed, paths["failed"])
    write_csv(stats, paths["stats"])
    write_csv(matrix, paths["matrix"])
    if show:
        print(f"{len(summary)} Läufe · nicht ok: {len(failed)} → " + ", ".join(p.name for p in paths.values()))
        with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
            display(HTML("<h4>Ergebnis je Experiment (Raten über den ersten gültigen Lauf je Aufgabe, 95-%-Wilson-KI)</h4>"))
            display(stats.set_index("experiment"))
            display(HTML("<h4>Aufgaben × Experimente (erster gültiger Lauf)</h4>"))
            show_task_matrix(matrix)
            display(HTML("<h4>Alle Läufe (chronologisch)</h4>"))
            display(summary.set_index("run_id"))
            display(HTML("<h4>Nicht-ok-Läufe (Volltexte: <code>python results\\show_runs.py --html --failed</code>)</h4>"))
            display(failed.set_index("Nr"))
    return {"summary": summary, "failed": failed, "stats": stats, "matrix": matrix, "frame": frame, "paths": paths}


export = export_results()

40 Läufe · nicht ok: 32 → run_summary.csv, failed_runs.csv, run_stats.csv, task_matrix.csv


,prompt_version,model,laeufe,infra_fehler,n_bewertet,weitere_laeufe,ok,ok_nach_korrektur,abbruch_dep,haengt_hold_message,keine_antwort,falsche_antwort,protokollfehler,rate_ok,rate_ok_ki_low,rate_ok_ki_high,rate_ok_inkl_korrektur,rate_ok_inkl_korrektur_ki_low,rate_ok_inkl_korrektur_ki_high
experiment,,,,,,,,,,,,,,,,,,,
E4,S1,qwen-3.8-180b-next,10,0,10,0,0,5,4,0,1,0,0,0.0,0.000,0.278,0.5,0.237,0.763
E1b,S0,qwen-3.8-180b-next,10,0,10,0,1,3,3,3,0,0,0,0.1,0.018,0.404,0.4,0.168,0.687
E7,S3,qwen-3.8-180b-next,10,0,10,0,3,2,4,0,1,0,0,0.3,0.108,0.603,0.5,0.237,0.763
E7.1,S3.1,qwen-3.8-180b-next,10,0,10,0,4,5,0,1,0,0,0,0.4,0.168,0.687,0.9,0.596,0.982


Aufgabe,E1b,E4,E7,E7.1
T1-01,ok_nach_korrektur,ok_nach_korrektur,abbruch_dep,ok_nach_korrektur
T1-02,haengt_hold_message,abbruch_dep,ok,ok
T1-03,ok_nach_korrektur,ok_nach_korrektur,abbruch_dep,ok
T1-04,haengt_hold_message,ok_nach_korrektur,abbruch_dep,ok
T1-05,ok_nach_korrektur,ok_nach_korrektur,ok,ok_nach_korrektur
T1-06,abbruch_dep,ok_nach_korrektur,ok,ok_nach_korrektur
T1-07,haengt_hold_message,abbruch_dep,abbruch_dep,ok_nach_korrektur
T1-08,abbruch_dep,abbruch_dep,keine_antwort,haengt_hold_message
T1-09,abbruch_dep,abbruch_dep,ok_nach_korrektur,ok_nach_korrektur
T1-10,ok,keine_antwort,ok_nach_korrektur,ok


,started_at,experiment,prompt_version,model,seed,item_id,question,soll,antwort,ergebnis,fehlgeschlagen_in_stage,grund,korrektur,leak,codes
run_id,,,,,,,,,,,,,,,
20261116T094141-1543de,2026-11-16T08:41:41.957+00:00,E4,S1,qwen-3.8-180b-next,101,T1-01,Please calculate 10+(2*4),18,18,ok_nach_korrektur,2,Stage 3: Tool-Call erst nach Korrektur-Prompt (Stage 2: Hold-Message),ja,nein,S2:A-OMIT S2:K-CORR
20261116T094157-af60e9,2026-11-16T08:41:57.327+00:00,E4,S1,qwen-3.8-180b-next,102,T1-02,Please calculate 13+(6*7),55,NaN,abbruch_dep,1,"Stage 1: ADD mit <DEP> vor dem ersten Ergebnis (<ADD x1=""13"" x2=""<DEP>""/>, im Antwortkanal) → Harness-Abbruch",nein,ja,S1:A-LEAK S1:A-WRONG
20261116T094158-18cfe2,2026-11-16T08:41:58.955+00:00,E4,S1,qwen-3.8-180b-next,103,T1-03,Please calculate 21+(4*9),57,57,ok_nach_korrektur,3,Stage 4: Endantwort erst nach Korrektur-Prompt (Stage 3: Hold-Message),ja,nein,S3:K-CORR
20261116T094209-4f1093,2026-11-16T08:42:09.984+00:00,E4,S1,qwen-3.8-180b-next,104,T1-04,Please calculate 17+(6*8),65,65,ok_nach_korrektur,3,Stage 4: Endantwort erst nach Korrektur-Prompt (Stage 3: keine Antwort),ja,nein,S3:K-CORR
20261116T094223-6c7c8e,2026-11-16T08:42:23.420+00:00,E4,S1,qwen-3.8-180b-next,105,T1-05,Please calculate 14+(7*9),77,77,ok_nach_korrektur,3,Stage 4: Endantwort erst nach Korrektur-Prompt (Stage 3: keine Antwort),ja,ja,S2:A-LEAK S3:K-CORR
20261116T094241-1da690,2026-11-16T08:42:41.375+00:00,E4,S1,qwen-3.8-180b-next,106,T1-06,Please calculate 26+(4*8),58,58,ok_nach_korrektur,3,Stage 4: Endantwort erst nach Korrektur-Prompt (Stage 3: Hold-Message),ja,nein,S2:B-SELF S3:K-CORR
20261116T094301-b2bc5f,2026-11-16T08:43:01.896+00:00,E4,S1,qwen-3.8-180b-next,107,T1-07,Please calculate 19+(6*9),73,NaN,abbruch_dep,1,"Stage 1: ADD mit <DEP> vor dem ersten Ergebnis (<ADD x1=""19"" x2=""<DEP>""/>, im Antwortkanal) → Harness-Abbruch",nein,ja,S1:A-LEAK S1:A-WRONG
20261116T094303-5f7045,2026-11-16T08:43:03.560+00:00,E4,S1,qwen-3.8-180b-next,108,T1-08,Please calculate 32+(7*8),88,NaN,abbruch_dep,1,"Stage 1: ADD mit <DEP> vor dem ersten Ergebnis (<ADD x1=""32"" x2=""<DEP>""/>, im Antwortkanal) → Harness-Abbruch",nein,ja,S1:A-LEAK S1:A-WRONG
20261116T094306-df0b5f,2026-11-16T08:43:06.032+00:00,E4,S1,qwen-3.8-180b-next,109,T1-09,Please calculate 11+(8*9),83,NaN,abbruch_dep,1,"Stage 1: ADD mit <DEP> vor dem ersten Ergebnis (<ADD x1=""11"" x2=""<DEP>""/>, im Antwortkanal) → Harness-Abbruch",nein,ja,S1:A-LEAK S1:A-WRONG


,Zeit,Experiment,Prompt,Aufgabe,Erwartet,Antwort,Ergebnis,gescheitert in Stufe,Code,Grund
Nr,,,,,,,,,,
0,16.11.2026 09:41,E4,S1,T1-01 · 10+(2*4),18,18,ok_nach_korrektur,2,A-OMIT K-CORR,Stage 3: Tool-Call erst nach Korrektur-Prompt (Stage 2: Hold-Message)
1,16.11.2026 09:41,E4,S1,T1-02 · 13+(6*7),55,NaN,abbruch_dep,1,A-LEAK A-WRONG,"Stage 1: ADD mit <DEP> vor dem ersten Ergebnis (<ADD x1=""13"" x2=""<DEP>""/>, im A…"
2,16.11.2026 09:41,E4,S1,T1-03 · 21+(4*9),57,57,ok_nach_korrektur,3,K-CORR,Stage 4: Endantwort erst nach Korrektur-Prompt (Stage 3: Hold-Message)
3,16.11.2026 09:42,E4,S1,T1-04 · 17+(6*8),65,65,ok_nach_korrektur,3,K-CORR,Stage 4: Endantwort erst nach Korrektur-Prompt (Stage 3: keine Antwort)
4,16.11.2026 09:42,E4,S1,T1-05 · 14+(7*9),77,77,ok_nach_korrektur,3,K-CORR,Stage 4: Endantwort erst nach Korrektur-Prompt (Stage 3: keine Antwort)
5,16.11.2026 09:42,E4,S1,T1-06 · 26+(4*8),58,58,ok_nach_korrektur,3,K-CORR,Stage 4: Endantwort erst nach Korrektur-Prompt (Stage 3: Hold-Message)
6,16.11.2026 09:43,E4,S1,T1-07 · 19+(6*9),73,NaN,abbruch_dep,1,A-LEAK A-WRONG,"Stage 1: ADD mit <DEP> vor dem ersten Ergebnis (<ADD x1=""19"" x2=""<DEP>""/>, im A…"
7,16.11.2026 09:43,E4,S1,T1-08 · 32+(7*8),88,NaN,abbruch_dep,1,A-LEAK A-WRONG,"Stage 1: ADD mit <DEP> vor dem ersten Ergebnis (<ADD x1=""32"" x2=""<DEP>""/>, im A…"
8,16.11.2026 09:43,E4,S1,T1-09 · 11+(8*9),83,NaN,abbruch_dep,1,A-LEAK A-WRONG,"Stage 1: ADD mit <DEP> vor dem ersten Ergebnis (<ADD x1=""11"" x2=""<DEP>""/>, im A…"


## 7. Laborbuch

*Gliederungspunkt.* Abweichungen, Auffälligkeiten, offene Punkte – mit Datum.

## 8. Interaktive Erkundung (Playground)

Hier lässt sich eine einzelne Frage frei durch den unveränderten Harness schicken. In **Phase −1** läuft nur ein
Request: Der Stream wird beim ersten leeren Tool-Tag (z. B. `<ADD x1="4" x2="9"/>`) abgebrochen, der Harness
rechnet das Ergebnis aus, schickt es aber nicht zurück – man sieht also, ob und welchen ersten Call das Modell
vorschlägt. In **Phase −2** wird dieses Ergebnis in einer `<RESULTS>`-Nachricht veröffentlicht und das
unterbrochene Thinking mit dem Id-Verweis als Resume-Präfix fortgesetzt; man sieht den zweiten Call und ob er den
Serverwert verwendet. **Phase −3** veröffentlicht auch das zweite Ergebnis und fordert dann ohne Tool-Stopp die
Endantwort an, die nur aus Serverwerten stammen sollte. Bleibt ein Tag aus, endet der Lauf mit der Halteantwort;
ab Phase −2 schickt der Harness dann einmal einen Korrektur-Prompt.

Playground-Läufe landen nur in `results/playground.jsonl` (`experiment = "playground"`), nie in
`results/runs.jsonl`, und zählen nicht zu den Experimenten.

In [ ]:
QUESTION = "Please calculate 7*(4+9)"
PHASE = -3                  # -1, -2 or -3
MODEL = "qwen-3.5-35b"      # e.g. "qwen-3.8-180b-next"
PROMPT_VERSION = "S0"       # key of PROMPT_PATHS
TEMPERATURE = 0.1
SEED = None                 # None: random seed for this run, stored in the record
RUN = False                 # True sends requests to the LLM endpoint

In [ ]:
_missing = [name for name in ("run_logged", "show_run", "PLAYGROUND_PATH") if name not in globals()]
if _missing:
    raise RuntimeError("Einrichtung fehlt: Zellen 1.1–1.7 ausführen (1.8 nicht nötig).")

import random


def playground_run(question, phase, model, prompt_version, temperature, seed, run,
                   cfg=None, out_path=None):
    """Summarise what would be sent; only with run=True send it via run_logged and return the record."""
    out_path = Path(out_path or PLAYGROUND_PATH)
    if phase not in (-1, -2, -3):
        raise ValueError(f"PHASE must be -1, -2 or -3, got {phase!r}")
    if prompt_version not in PROMPT_PATHS:
        raise ValueError(f"unknown PROMPT_VERSION {prompt_version!r}; available: {sorted(PROMPT_PATHS)}")
    if out_path.resolve() == RUNS_PATH.resolve():
        raise ValueError("the playground never writes to results/runs.jsonl")
    cfg = model_cfg if cfg is None else cfg
    prompt = load_prompt(prompt_version)
    if seed is None:
        seed = random.randrange(2**31)
    print("playground run" + ("" if run else " (RUN = False: nothing is sent)"))
    print(f"  model      : {model}")
    print(f"  endpoint   : {cfg['endpoint'] if cfg is not None else '(no API key loaded)'}")
    print(f"  phase      : {phase}")
    print(f"  question   : {question}")
    print(f"  prompt     : {prompt['version']} ({prompt['path']}, sha256 {prompt['sha256'][:12]}…)")
    print(f"  temperature: {temperature} | max_tokens: {MAX_TOKENS} | seed: {seed}")
    print(f"  output     : {out_path.relative_to(ROOT).as_posix() if out_path.is_relative_to(ROOT) else out_path}")
    if not run:
        print("no request sent; set RUN = True to run")
        return None
    if cfg is None:
        raise RuntimeError("no API key loaded (section 1.3): a live run is not possible")
    run_cfg = SafeCfg(cfg)
    run_cfg["model"] = model
    record = run_logged(
        run_cfg, prompt, question, abs(phase),
        experiment="playground",
        seed=seed,
        out_path=out_path,
        temperature=temperature,
        note="playground: exploratory, not part of the experiment matrix",
    )
    show_run(record)
    return record


playground_record = playground_run(QUESTION, PHASE, MODEL, PROMPT_VERSION, TEMPERATURE, SEED, RUN)

Nach einem Lauf `RUN` wieder auf `False` stellen, damit „Run All“ keine Requests auslöst.